In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs into DataFrames (stored for later steps)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Define timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Helper to produce timezone-aware min/max strings from a dataframe timestamp column
def tz_min_max(df, ts_col='timestamp'):
    if df.shape[0] == 0:
        return pd.Series({'min_timestamp': None, 'max_timestamp': None})
    ts_min = pd.to_datetime(df[ts_col].min(), unit='s', utc=True).tz_convert(tz)
    ts_max = pd.to_datetime(df[ts_col].max(), unit='s', utc=True).tz_convert(tz)
    # Compact string format
    return pd.Series({
        'min_timestamp': ts_min.strftime('%Y-%m-%d %H:%M:%S%z'),
        'max_timestamp': ts_max.strftime('%Y-%m-%d %H:%M:%S%z')
    })

# 1) metric.csv summary
metric_kpi_counts = (
    metric_df['kpi_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'kpi_name', 'kpi_name': 'count'})
    .head(50)
)
metric_top_cmdb = (
    metric_df['cmdb_id']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'})
    .head(20)
)
metric_time_range = tz_min_max(metric_df, 'timestamp')

# 2) trace.csv summary
trace_trace_counts = (
    trace_df['trace_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'trace_name', 'trace_name': 'count'})
    .head(50)
)
trace_top_cmdb = (
    trace_df['cmdb_id']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'})
    .head(20)
)
trace_time_range = tz_min_max(trace_df, 'timestamp')

# 3) log.csv summary
log_log_counts = (
    log_df['log_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'log_name', 'log_name': 'count'})
    .head(50)
)
log_top_cmdb = (
    log_df['cmdb_id']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'})
    .head(20)
)
log_time_range = tz_min_max(log_df, 'timestamp')

# 4) error_logs.csv summary
error_total_rows = int(error_df.shape[0])
error_cmdb_counts = (
    error_df['cmdb_id']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'})
    .head(20)
)
error_time_range = tz_min_max(error_df, 'timestamp')

# Prepare up to 10 sample rows with converted timestamp string (compact)
if error_df.shape[0] > 0:
    error_samples = error_df[['timestamp', 'cmdb_id', 'message']].head(10).copy()
    error_samples['timestamp'] = pd.to_datetime(error_samples['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S%z')
else:
    error_samples = error_df[['timestamp', 'cmdb_id', 'message']].head(0)

# Final compact outputs (variables to be displayed)
metric_kpi_counts, metric_top_cmdb, metric_time_range, trace_trace_counts, trace_top_cmdb, trace_time_range, log_log_counts, log_top_cmdb, log_time_range, error_total_rows, error_cmdb_counts, error_time_range, error_samples

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry (UTC+8):\n"
    "- metric.csv: Top KPI prefixes are many container.node-5.* metrics (highest counts shown), dataset covers 2022-03-20 09:00:00+0800 to 2022-03-20 09:29:00+0800. "
    "Top 20 components by record count include adservice, adservice2, frontend pods, checkoutservice pods and productcatalogservice pods (adservice/adservice2 have the most records).\n"
    "- trace.csv: Most frequent trace names include trace.from_frontend-0.duration_mean / duration_p95 / error_rate / row_count (highest counts ~584). "
    "Top trace components include frontend-0, frontend-2, frontend-1. Time range: 2022-03-20 09:00:00+0800 to 2022-03-20 09:29:00+0800.\n"
    "- log.csv: Only two log metrics present (log.error_count and log.row_count, both count=873). Top components each have ~60 log records (many services/pods listed). "
    "Time range: 2022-03-20 09:00:00+0800 to 2022-03-20 09:29:00+0800.\n"
    "- error_logs.csv: Total rows = 558. Error rows concentrated on frontend pods: frontend-2 (231), frontend-0 (229), frontend-1 (98). "
    "Error timestamps span 2022-03-20 09:00:01+0800 to 2022-03-20 09:29:56+0800. Sample messages show repeated warnings like 'failed to retrieve...'.\n\n"
    "Overall observation: telemetry covers the same 30-minute window. Metrics and traces show heavy activity for adservice/adservice2 and frontend pods; error logs are dominated by frontend pods with repeated warning messages. "
    "If you want, I can (a) show the full top-N lists, (b) expand error log samples, or (c) compute per-component error rates or anomaly candidates next."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                                count  count
0   container.node-5.container_network_receive_pac...   1050
1   container.node-5.container_network_transmit_MB...   1050
2   container.node-5.container_network_receive_MB....   1050
3   container.node-5.container_network_transmit_pa...   1050
4   container.node-5.container_network_transmit_er...   1050
..                                                ...    ...
45        container.node-5.container_cpu_user_seconds    990
46        container.node-5.container_file_descriptors    990
47    container.node-5.container_cpu_load_average_10s    990
48  container.node-5.container_fs_io_current./dev/...    990
49  container.node-5.container_fs_inodes_free./dev...    990

[50 rows x 2 columns],                        count  count
0                  adservice  10230
1                 adservice2   9990
2                 frontend-0   4680
3                 frontend-1   4560
4                frontend2-0   4560
5                 frontend-2   4140
6          checkoutservice-0   3540
7          checkoutservice-1   3540
8          checkoutservice-2   3300
9         checkoutservice2-0   3300
10   productcatalogservice-2   3240
11   productcatalogservice-0   3120
12   productcatalogservice-1   3120
13             cartservice-1   2760
14             cartservice-0   2640
15            cartservice2-0   2640
16              adservice2-0   2580
17               adservice-0   2580
18             cartservice-2   2520
19  productcatalogservice2-0   2520, min_timestamp    2022-03-20 09:00:00+0800
max_timestamp    2022-03-20 09:29:00+0800
dtype: object,                                           count  count
0           trace.from_frontend-0.duration_mean    584
1            trace.from_frontend-0.duration_p95    584
2              trace.from_frontend-0.error_rate    584
3               trace.from_frontend-0.row_count    584
4               trace.from_frontend-2.row_count    583
..                                          ...    ...
45          trace.from_frontend2-0.duration_p95    210
46            trace.from_frontend2-0.error_rate    210
47             trace.from_frontend2-0.row_count    210
48  trace.from_checkoutservice2-0.duration_mean    180
49   trace.from_checkoutservice2-0.duration_p95    180

[50 rows x 2 columns],                       count  count
0                frontend-0   2576
1                frontend-2   2572
2                frontend-1   2360
3         checkoutservice-1   1720
4         checkoutservice-2   1656
5         checkoutservice-0   1572
6   productcatalogservice-2   1092
7               frontend2-0   1080
8        checkoutservice2-0   1080
9   productcatalogservice-0   1076
10  productcatalogservice-1   1064
11  recommendationservice-0    840
12  recommendationservice-1    840
13  recommendationservice-2    840
14            cartservice-0    728
15            cartservice-1    724
16            cartservice-2    712
17        currencyservice-2    652
18        currencyservice-1    648
19        currencyservice-0    648, min_timestamp    2022-03-20 09:00:00+0800
max_timestamp    2022-03-20 09:29:00+0800
dtype: object,              count  count
0  log.error_count    873
1    log.row_count    873,                       count  count
0               adservice-0     60
1               adservice-1     60
2               adservice-2     60
3             cartservice-0     60
4             cartservice-1     60
5             cartservice-2     60
6         currencyservice-1     60
7                frontend-0     60
8         currencyservice-0     60
9         currencyservice-2     60
10  recommendationservice-0     60
11  productcatalogservice-1     60
12  productcatalogservice-0     60
13               frontend-2     60
14               frontend-1     60
15        shippingservice-1     60
16        shippingservice-0     60
17        shippingservice-2     60
18  recommendationservice-2     60
19  recommendationservice-1     60, min_timestamp    2022-03-20 09:00:00+0800
max_timestamp    2022-03-20 09:29:00+0800
dtype: object, 558,         count  count
0  frontend-2    231
1  frontend-0    229
2  frontend-1     98, min_timestamp    2022-03-20 09:00:01+0800
max_timestamp    2022-03-20 09:29:56+0800
dtype: object,                   timestamp     cmdb_id                                            message
0  2022-03-20 09:00:01+0800  frontend-0  severity: warning, message: failed to retrieve...
1  2022-03-20 09:00:01+0800  frontend-2  severity: warning, message: failed to retrieve...
2  2022-03-20 09:00:07+0800  frontend-0  severity: warning, message: failed to retrieve...
3  2022-03-20 09:00:07+0800  frontend-2  severity: warning, message: failed to retrieve...
4  2022-03-20 09:00:13+0800  frontend-2  severity: warning, message: failed to retrieve...
5  2022-03-20 09:00:15+0800  frontend-0  severity: warning, message: failed to retrieve...
6  2022-03-20 09:00:17+0800  frontend-2  severity: warning, message: failed to retrieve...
7  2022-03-20 09:00:19+0800  frontend-0  severity: warning, message: failed to retrieve...
8  2022-03-20 09:00:24+0800  frontend-2  severity: warning, message: failed to retrieve...
9  2022-03-20 09:00:29+0800  frontend-2  severity: warning, message: failed to retrieve...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing metric_df loaded previously
tz = pytz.timezone('Asia/Shanghai')
frontend_ids = ['frontend-0', 'frontend-1', 'frontend-2']

# Full series for frontend pods (no time filtering) - used to compute global P95 and total points
metric_frontends_full = metric_df[metric_df['cmdb_id'].isin(frontend_ids)].copy()

# Compute global P95 and total points per (cmdb_id, kpi_name) using full available series
grp_full = metric_frontends_full.groupby(['cmdb_id', 'kpi_name'])['value']
global_p95_series = grp_full.quantile(0.95)
total_points_series = grp_full.size()

df_global = (
    pd.DataFrame({
        'global_p95': global_p95_series,
        'total_points_in_full_series': total_points_series
    })
    .reset_index()
)

# Prepare incident-window filtered data for the same three frontends
start_dt = tz.localize(pd.Timestamp('2022-03-20 09:00:00'))
end_dt = tz.localize(pd.Timestamp('2022-03-20 09:30:00'))  # exclusive upper bound
metric_frontends_incident = metric_frontends_full.copy()
metric_frontends_incident['dt'] = pd.to_datetime(metric_frontends_incident['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
# Filter to incident window [start_dt, end_dt)
metric_frontends_incident = metric_frontends_incident[
    (metric_frontends_incident['dt'] >= start_dt) &
    (metric_frontends_incident['dt'] < end_dt)
].copy()

# Join global thresholds onto incident rows so we can compute above-P95 counts
metric_incident_join = metric_frontends_incident.merge(df_global, on=['cmdb_id', 'kpi_name'], how='left')

# Mark rows above global p95; create dt_above which is dt only when above, else NaT
metric_incident_join['above_p95'] = metric_incident_join['value'] > metric_incident_join['global_p95']
metric_incident_join['dt_above'] = metric_incident_join['dt'].where(metric_incident_join['above_p95'], pd.NaT)

# Aggregate incident stats per (cmdb_id, kpi_name)
incident_agg = (
    metric_incident_join
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        incident_points_count_above_p95 = ('above_p95', 'sum'),
        earliest_timestamp_above = ('dt_above', 'min'),
        max_value_in_incident_window = ('value', 'max'),
        incident_points_total = ('value', 'count')
    )
    .reset_index()
)

# Merge global info with incident aggregates to ensure pairs with zero incident rows are included
result_df = df_global.merge(incident_agg, on=['cmdb_id', 'kpi_name'], how='left')

# Fill NaNs for groups without incident rows
result_df['incident_points_count_above_p95'] = result_df['incident_points_count_above_p95'].fillna(0).astype(int)
# earliest_timestamp_above keep NaT for none
# max_value_in_incident_window may be NaN; leave as is
result_df['incident_points_total'] = result_df['incident_points_total'].fillna(0).astype(int)

# Format earliest timestamp column as string, keep nulls as None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return x.strftime('%Y-%m-%d %H:%M:%S%z')

result_df['earliest_timestamp_above'] = result_df['earliest_timestamp_above'].apply(fmt_ts)

# Select and order columns as requested
final_cols = [
    'cmdb_id',
    'kpi_name',
    'global_p95',
    'total_points_in_full_series',
    'incident_points_count_above_p95',
    'earliest_timestamp_above',
    'max_value_in_incident_window'
]
final_df = result_df[final_cols].copy()

# Sort by incident_points_count_above_p95 desc, then earliest timestamp asc (None/NaN last)
# For sorting by earliest timestamp asc, convert None to large timestamp so they appear last
final_df['__earliest_sort'] = pd.to_datetime(final_df['earliest_timestamp_above'], format='%Y-%m-%d %H:%M:%S%z', utc=True, errors='coerce')
final_df = final_df.sort_values(by=['incident_points_count_above_p95', '__earliest_sort'], ascending=[False, True], na_position='last')
final_df = final_df.drop(columns='__earliest_sort')

# Limit to top 50 rows for compact output
final_df_top50 = final_df.head(50)

# Store intermediates for later steps in kernel variables (as required)
metric_frontends_full, df_global, metric_frontends_incident, metric_incident_join, incident_agg, final_df_top50

```
Out[2]:
```
summary = (
    "Frontend metric P95 analysis (computed on full series; incident window 2022-03-20 09:00–09:30 UTC+8):\n"
    "- Processed 3 pods (frontend-0, frontend-1, frontend-2) and 446 (cmdb_id, kpi_name) pairs. Each pair has 30 full-series points (global statistics computed on full series).\n"
    "- Most KPI series did NOT exceed their global P95 inside the incident window. Exceedances are few and small: the highest incident_points_count_above_p95 observed is 2 (many KPIs), with several KPIs showing 1, and many 0.\n"
    "- Example notable exceedances (top rows):\n"
    "  * frontend-0 mesh.destination.unknown.frontend.istio_requests*: global_p95=35.775, incident_points_above=2, earliest exceed=2022-03-20 09:00:00+0800, max_in_incident=36.0\n"
    "  * frontend-0 mesh.source.frontend.checkoutservice.istio_req*: global_p95=4578.75, incident_points_above=2, earliest=2022-03-20 09:00:00+0800, max=4950.0\n"
    "  * frontend-0 container.node-5.container_cpu_cfs_periods: global_p95=114.25, incident_points_above=2, earliest=2022-03-20 09:06:00+0800, max=123.0\n"
    "  * frontend-0 container.node-5.container_cpu_usage_seconds: global_p95≈0.30079, incident_points_above=2, earliest=2022-03-20 09:06:00+0800, max≈0.30750\n"
    "  * frontend-2 mesh.source...istio_res: global_p95=1860.0, incident_points_above=1, earliest=2022-03-20 09:11:00+0800, max=1937.5\n"
    "  * frontend-1 container.node-5.container_memory_usage_MB: global_p95≈25.165, incident_points_above=2, earliest=2022-03-20 09:05:00+0800, max≈25.340\n"
    "- In short: during the incident window a small subset of frontend metrics briefly exceeded their global P95 (mostly 1–2 points per KPI). Key areas with exceedances include a few mesh/istio request metrics and some container CPU/memory-related metrics on the frontend pods.\n"
    "If you want, I can return the full top-50 compact table of (cmdb_id, kpi_name, global_p95, total_points, incident_count_above_p95, earliest_ts, max_in_incident) for inspection."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         timestamp     cmdb_id                                           kpi_name       value
2426    1647738000  frontend-0         container.node-5.container_cpu_cfs_periods   103.00000
2427    1647738000  frontend-0  container.node-5.container_cpu_cfs_throttled_p...     0.00000
2428    1647738000  frontend-0  container.node-5.container_cpu_cfs_throttled_s...     0.00000
2429    1647738000  frontend-0    container.node-5.container_cpu_load_average_10s     0.00000
2430    1647738000  frontend-0      container.node-5.container_cpu_system_seconds     0.07000
...            ...         ...                                                ...         ...
139761  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_req...     0.00735
139762  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_req...     7.50000
139763  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_req...     7.50000
139764  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_res...  1085.00000
139765  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_res...     7.50000

[13380 rows x 4 columns],         cmdb_id                                           kpi_name   global_p95  total_points_in_full_series
0    frontend-0         container.node-5.container_cpu_cfs_periods   114.250000                           30
1    frontend-0  container.node-5.container_cpu_cfs_throttled_p...     0.000000                           30
2    frontend-0  container.node-5.container_cpu_cfs_throttled_s...     0.000000                           30
3    frontend-0    container.node-5.container_cpu_load_average_10s     0.000000                           30
4    frontend-0      container.node-5.container_cpu_system_seconds     0.090000                           30
..          ...                                                ...          ...                          ...
441  frontend-2  mesh.source.frontend.shippingservice.istio_req...     0.012364                           30
442  frontend-2  mesh.source.frontend.shippingservice.istio_req...    11.500000                           30
443  frontend-2  mesh.source.frontend.shippingservice.istio_req...    11.500000                           30
444  frontend-2  mesh.source.frontend.shippingservice.istio_res...  1860.000000                           30
445  frontend-2  mesh.source.frontend.shippingservice.istio_res...    11.500000                           30

[446 rows x 4 columns],          timestamp     cmdb_id                                           kpi_name       value                        dt
2426    1647738000  frontend-0         container.node-5.container_cpu_cfs_periods   103.00000 2022-03-20 09:00:00+08:00
2427    1647738000  frontend-0  container.node-5.container_cpu_cfs_throttled_p...     0.00000 2022-03-20 09:00:00+08:00
2428    1647738000  frontend-0  container.node-5.container_cpu_cfs_throttled_s...     0.00000 2022-03-20 09:00:00+08:00
2429    1647738000  frontend-0    container.node-5.container_cpu_load_average_10s     0.00000 2022-03-20 09:00:00+08:00
2430    1647738000  frontend-0      container.node-5.container_cpu_system_seconds     0.07000 2022-03-20 09:00:00+08:00
...            ...         ...                                                ...         ...                       ...
139761  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_req...     0.00735 2022-03-20 09:29:00+08:00
139762  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_req...     7.50000 2022-03-20 09:29:00+08:00
139763  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_req...     7.50000 2022-03-20 09:29:00+08:00
139764  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_res...  1085.00000 2022-03-20 09:29:00+08:00
139765  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_res...     7.50000 2022-03-20 09:29:00+08:00

[13380 rows x 5 columns],         timestamp     cmdb_id                                           kpi_name       value                        dt   global_p95  total_points_in_full_series  above_p95 dt_above
0      1647738000  frontend-0         container.node-5.container_cpu_cfs_periods   103.00000 2022-03-20 09:00:00+08:00   114.250000                           30      False      NaT
1      1647738000  frontend-0  container.node-5.container_cpu_cfs_throttled_p...     0.00000 2022-03-20 09:00:00+08:00     0.000000                           30      False      NaT
2      1647738000  frontend-0  container.node-5.container_cpu_cfs_throttled_s...     0.00000 2022-03-20 09:00:00+08:00     0.000000                           30      False      NaT
3      1647738000  frontend-0    container.node-5.container_cpu_load_average_10s     0.00000 2022-03-20 09:00:00+08:00     0.000000                           30      False      NaT
4      1647738000  frontend-0      container.node-5.container_cpu_system_seconds     0.07000 2022-03-20 09:00:00+08:00     0.090000                           30      False      NaT
...           ...         ...                                                ...         ...                       ...          ...                          ...        ...      ...
13375  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_req...     0.00735 2022-03-20 09:29:00+08:00     0.012364                           30      False      NaT
13376  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_req...     7.50000 2022-03-20 09:29:00+08:00    11.500000                           30      False      NaT
13377  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_req...     7.50000 2022-03-20 09:29:00+08:00    11.500000                           30      False      NaT
13378  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_res...  1085.00000 2022-03-20 09:29:00+08:00  1860.000000                           30      False      NaT
13379  1647739740  frontend-2  mesh.source.frontend.shippingservice.istio_res...     7.50000 2022-03-20 09:29:00+08:00    11.500000                           30      False      NaT

[13380 rows x 9 columns],         cmdb_id                                           kpi_name  incident_points_count_above_p95  earliest_timestamp_above  max_value_in_incident_window  incident_points_total
0    frontend-0         container.node-5.container_cpu_cfs_periods                                2 2022-03-20 09:06:00+08:00                      123.0000                     30
1    frontend-0  container.node-5.container_cpu_cfs_throttled_p...                                0                       NaT                        0.0000                     30
2    frontend-0  container.node-5.container_cpu_cfs_throttled_s...                                0                       NaT                        0.0000                     30
3    frontend-0    container.node-5.container_cpu_load_average_10s                                0                       NaT                        0.0000                     30
4    frontend-0      container.node-5.container_cpu_system_seconds                                1 2022-03-20 09:06:00+08:00                        0.0950                     30
..          ...                                                ...                              ...                       ...                           ...                    ...
441  frontend-2  mesh.source.frontend.shippingservice.istio_req...                                2 2022-03-20 09:00:00+08:00                        0.0126                     30
442  frontend-2  mesh.source.frontend.shippingservice.istio_req...                                1 2022-03-20 09:11:00+08:00                       12.0000                     30
443  frontend-2  mesh.source.frontend.shippingservice.istio_req...                                1 2022-03-20 09:11:00+08:00                       12.0000                     30
444  frontend-2  mesh.source.frontend.shippingservice.istio_res...                                1 2022-03-20 09:11:00+08:00                     1937.5000                     30
445  frontend-2  mesh.source.frontend.shippingservice.istio_res...                                1 2022-03-20 09:11:00+08:00                       12.0000                     30

[446 rows x 6 columns],         cmdb_id                                           kpi_name   global_p95  total_points_in_full_series  incident_points_count_above_p95  earliest_timestamp_above  max_value_in_incident_window
73   frontend-0  mesh.destination.unknown.frontend.istio_reques...    35.775000                           30                                2  2022-03-20 09:00:00+0800                     36.000000
108  frontend-0  mesh.source.frontend.checkoutservice.istio_req...  4578.750000                           30                                2  2022-03-20 09:00:00+0800                   4950.000000
110  frontend-0  mesh.source.frontend.checkoutservice.istio_req...     0.106112                           30                                2  2022-03-20 09:00:00+0800                      0.110500
112  frontend-0  mesh.source.frontend.checkoutservice.istio_req...     2.775000                           30                                2  2022-03-20 09:00:00+0800                      3.000000
113  frontend-0  mesh.source.frontend.checkoutservice.istio_req...     2.775000                           30                                2  2022-03-20 09:00:00+0800                      3.000000
..          ...                                                ...          ...                          ...                              ...                       ...                           ...
194  frontend-1         container.node-5.container_memory_usage_MB    25.164844                           30                                2  2022-03-20 09:05:00+0800                     25.339844
195  frontend-1   container.node-5.container_memory_working_set_MB    25.117969                           30                                2  2022-03-20 09:05:00+0800                     25.292969
200  frontend-1  container.node-5.container_network_transmit_MB...     2.990145                           30                                2  2022-03-20 09:05:00+0800                      3.594130
0    frontend-0         container.node-5.container_cpu_cfs_periods   114.250000                           30                                2  2022-03-20 09:06:00+0800                    123.000000
5    frontend-0       container.node-5.container_cpu_usage_seconds     0.300790                           30                                2  2022-03-20 09:06:00+0800                      0.307497

[50 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse trace_df and tz from previous kernel state
# If not present, load trace.csv (safe-guard)
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and total points per (cmdb_id, trace_name) using full series (no time filtering)
grp_full = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_global_p95 = grp_full.quantile(0.95)
trace_total_points = grp_full.size()

df_trace_global = (
    pd.DataFrame({
        'global_p95': trace_global_p95,
        'total_points_in_full_series': trace_total_points
    })
    .reset_index()
)

# 2) Prepare incident-window filtered trace records (UTC+8)
start_dt = tz.localize(pd.Timestamp('2022-03-20 09:00:00'))
end_dt = tz.localize(pd.Timestamp('2022-03-20 09:30:00'))  # exclusive upper bound

trace_incident = trace_df.copy()
trace_incident['dt'] = pd.to_datetime(trace_incident['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_incident = trace_incident[(trace_incident['dt'] >= start_dt) & (trace_incident['dt'] < end_dt)].copy()

# 3) Join global thresholds to incident rows
trace_incident_join = trace_incident.merge(df_trace_global, on=['cmdb_id', 'trace_name'], how='left')

# 4) Mark above-P95 and earliest above timestamp
trace_incident_join['above_p95'] = trace_incident_join['value'] > trace_incident_join['global_p95']
trace_incident_join['dt_above'] = trace_incident_join['dt'].where(trace_incident_join['above_p95'], pd.NaT)

# 5) Aggregate incident stats per (cmdb_id, trace_name)
trace_incident_agg = (
    trace_incident_join
    .groupby(['cmdb_id', 'trace_name'])
    .agg(
        incident_points_count_above_p95 = ('above_p95', 'sum'),
        earliest_timestamp_above = ('dt_above', 'min'),
        max_value_in_incident_window = ('value', 'max'),
        incident_points_total = ('value', 'count')
    )
    .reset_index()
)

# 6) Merge with global info to include series that may have zero incident rows
trace_result = df_trace_global.merge(trace_incident_agg, on=['cmdb_id', 'trace_name'], how='left')

# 7) Fill NaNs for counts, keep earliest as None if missing
trace_result['incident_points_count_above_p95'] = trace_result['incident_points_count_above_p95'].fillna(0).astype(int)
trace_result['incident_points_total'] = trace_result['incident_points_total'].fillna(0).astype(int)

def fmt_ts(x):
    if pd.isna(x):
        return None
    return x.strftime('%Y-%m-%d %H:%M:%S%z')

trace_result['earliest_timestamp_above'] = trace_result['earliest_timestamp_above'].apply(fmt_ts)

# 8) Select required columns and sort as requested
final_cols = [
    'cmdb_id',
    'trace_name',
    'global_p95',
    'total_points_in_full_series',
    'incident_points_count_above_p95',
    'earliest_timestamp_above',
    'max_value_in_incident_window'
]
trace_final = trace_result[final_cols].copy()

# For sorting by earliest timestamp asc while placing None last, create helper sort key
trace_final['__earliest_sort'] = pd.to_datetime(trace_final['earliest_timestamp_above'], format='%Y-%m-%d %H:%M:%S%z', utc=True, errors='coerce')
trace_final = trace_final.sort_values(by=['incident_points_count_above_p95', '__earliest_sort'], ascending=[False, True], na_position='last')
trace_final = trace_final.drop(columns='__earliest_sort')

# 9) Limit to top 50 rows for compact output
trace_final_top50 = trace_final.head(50)

# Store intermediates for later steps
trace_global_p95, df_trace_global, trace_incident, trace_incident_join, trace_incident_agg, trace_final_top50

```
Out[3]:
```
summary = (
    "Trace P95 analysis (global P95 computed on full series; incident window 2022-03-20 09:00–09:30 UTC+8):\n"
    "- Processed 1,232 (cmdb_id, trace_name) series. Global P95 and total-points were computed from full series; incident-window stats used only records in the 09:00–09:30 window.\n"
    "- Most trace series did not substantially exceed their global P95 during the incident window. Exceedances are relatively rare and small: the highest incident_points_count_above_p95 observed in the top results is 2 (many series), with many series showing 1 or 0 exceedances.\n"
    "- Notable examples from the top results (illustrative):\n"
    "  * cartservice-0, trace.from_frontend-2.row_count — global_p95=13.55, incident_points_above=2, earliest exceed=2022-03-20 09:00:00+08:00, max_in_incident=15.0\n"
    "  * checkoutservice-1, trace.from_root.duration_p95 — global_p95≈0.003441, incident_points_above=2, earliest=2022-03-20 09:00:00+08:00\n"
    "  * frontend-0, several traces to productcatalogservice/adservice — multiple series with 2 exceedances around 09:00–09:01 and small increases over P95.\n"
    "- Overall conclusion: traces show occasional brief spikes (mostly 1–2 points) across multiple services (frontend, cartservice, checkoutservice, etc.), rather than a large sustained deviation in many series. These are candidates for targeted inspection (check corresponding traces/logs at the noted earliest timestamps).\n"
    "If you want, I can (a) return the full top-50 table, (b) list all series with >0 exceedances, or (c) correlate these trace spikes with metrics/logs at the same timestamps."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(cmdb_id             trace_name                             
adservice-0         trace.from_frontend-0.duration_mean         0.000017
                    trace.from_frontend-0.duration_p95          0.000032
                    trace.from_frontend-0.error_rate            0.000000
                    trace.from_frontend-0.row_count            10.000000
                    trace.from_frontend-1.duration_mean         0.000016
                                                                 ...    
shippingservice2-0  trace.from_checkoutservice2-0.row_count     4.000000
                    trace.from_frontend2-0.duration_mean        0.000075
                    trace.from_frontend2-0.duration_p95         0.000124
                    trace.from_frontend2-0.error_rate           0.000000
                    trace.from_frontend2-0.row_count           13.550000
Name: value, Length: 1232, dtype: float64,                  cmdb_id                               trace_name  global_p95  total_points_in_full_series
0            adservice-0      trace.from_frontend-0.duration_mean    0.000017                           30
1            adservice-0       trace.from_frontend-0.duration_p95    0.000032                           30
2            adservice-0         trace.from_frontend-0.error_rate    0.000000                           30
3            adservice-0          trace.from_frontend-0.row_count   10.000000                           30
4            adservice-0      trace.from_frontend-1.duration_mean    0.000016                           30
...                  ...                                      ...         ...                          ...
1227  shippingservice2-0  trace.from_checkoutservice2-0.row_count    4.000000                           30
1228  shippingservice2-0     trace.from_frontend2-0.duration_mean    0.000075                           30
1229  shippingservice2-0      trace.from_frontend2-0.duration_p95    0.000124                           30
1230  shippingservice2-0        trace.from_frontend2-0.error_rate    0.000000                           30
1231  shippingservice2-0         trace.from_frontend2-0.row_count   13.550000                           30

[1232 rows x 4 columns],         timestamp             cmdb_id                               trace_name      value                        dt
0      1647738000         adservice-0      trace.from_frontend-0.duration_mean   0.000013 2022-03-20 09:00:00+08:00
1      1647738000         adservice-0       trace.from_frontend-0.duration_p95   0.000016 2022-03-20 09:00:00+08:00
2      1647738000         adservice-0         trace.from_frontend-0.error_rate   0.000000 2022-03-20 09:00:00+08:00
3      1647738000         adservice-0          trace.from_frontend-0.row_count   6.000000 2022-03-20 09:00:00+08:00
4      1647738000         adservice-0      trace.from_frontend-1.duration_mean   0.000014 2022-03-20 09:00:00+08:00
...           ...                 ...                                      ...        ...                       ...
30147  1647739740  shippingservice2-0  trace.from_checkoutservice2-0.row_count   2.000000 2022-03-20 09:29:00+08:00
30148  1647739740  shippingservice2-0     trace.from_frontend2-0.duration_mean   0.000071 2022-03-20 09:29:00+08:00
30149  1647739740  shippingservice2-0      trace.from_frontend2-0.duration_p95   0.000119 2022-03-20 09:29:00+08:00
30150  1647739740  shippingservice2-0        trace.from_frontend2-0.error_rate   0.000000 2022-03-20 09:29:00+08:00
30151  1647739740  shippingservice2-0         trace.from_frontend2-0.row_count  15.000000 2022-03-20 09:29:00+08:00

[30152 rows x 5 columns],         timestamp             cmdb_id                               trace_name      value                        dt  global_p95  total_points_in_full_series  above_p95                  dt_above
0      1647738000         adservice-0      trace.from_frontend-0.duration_mean   0.000013 2022-03-20 09:00:00+08:00    0.000017                           30      False                       NaT
1      1647738000         adservice-0       trace.from_frontend-0.duration_p95   0.000016 2022-03-20 09:00:00+08:00    0.000032                           30      False                       NaT
2      1647738000         adservice-0         trace.from_frontend-0.error_rate   0.000000 2022-03-20 09:00:00+08:00    0.000000                           30      False                       NaT
3      1647738000         adservice-0          trace.from_frontend-0.row_count   6.000000 2022-03-20 09:00:00+08:00   10.000000                           30      False                       NaT
4      1647738000         adservice-0      trace.from_frontend-1.duration_mean   0.000014 2022-03-20 09:00:00+08:00    0.000016                           30      False                       NaT
...           ...                 ...                                      ...        ...                       ...         ...                          ...        ...                       ...
30147  1647739740  shippingservice2-0  trace.from_checkoutservice2-0.row_count   2.000000 2022-03-20 09:29:00+08:00    4.000000                           30      False                       NaT
30148  1647739740  shippingservice2-0     trace.from_frontend2-0.duration_mean   0.000071 2022-03-20 09:29:00+08:00    0.000075                           30      False                       NaT
30149  1647739740  shippingservice2-0      trace.from_frontend2-0.duration_p95   0.000119 2022-03-20 09:29:00+08:00    0.000124                           30      False                       NaT
30150  1647739740  shippingservice2-0        trace.from_frontend2-0.error_rate   0.000000 2022-03-20 09:29:00+08:00    0.000000                           30      False                       NaT
30151  1647739740  shippingservice2-0         trace.from_frontend2-0.row_count  15.000000 2022-03-20 09:29:00+08:00   13.550000                           30       True 2022-03-20 09:29:00+08:00

[30152 rows x 9 columns],                  cmdb_id                               trace_name  incident_points_count_above_p95  earliest_timestamp_above  max_value_in_incident_window  incident_points_total
0            adservice-0      trace.from_frontend-0.duration_mean                                2 2022-03-20 09:17:00+08:00                      0.000035                     30
1            adservice-0       trace.from_frontend-0.duration_p95                                2 2022-03-20 09:17:00+08:00                      0.000128                     30
2            adservice-0         trace.from_frontend-0.error_rate                                0                       NaT                      0.000000                     30
3            adservice-0          trace.from_frontend-0.row_count                                1 2022-03-20 09:25:00+08:00                     11.000000                     30
4            adservice-0      trace.from_frontend-1.duration_mean                                2 2022-03-20 09:01:00+08:00                      0.000017                     30
...                  ...                                      ...                              ...                       ...                           ...                    ...
1227  shippingservice2-0  trace.from_checkoutservice2-0.row_count                                1 2022-03-20 09:15:00+08:00                      6.000000                     30
1228  shippingservice2-0     trace.from_frontend2-0.duration_mean                                2 2022-03-20 09:14:00+08:00                      0.000091                     30
1229  shippingservice2-0      trace.from_frontend2-0.duration_p95                                2 2022-03-20 09:08:00+08:00                      0.000198                     30
1230  shippingservice2-0        trace.from_frontend2-0.error_rate                                0                       NaT                      0.000000                     30
1231  shippingservice2-0         trace.from_frontend2-0.row_count                                2 2022-03-20 09:00:00+08:00                     15.000000                     30

[1232 rows x 6 columns],                 cmdb_id                                      trace_name  global_p95  total_points_in_full_series  incident_points_count_above_p95  earliest_timestamp_above  max_value_in_incident_window
63        cartservice-0                 trace.from_frontend-2.row_count   13.550000                           30                                2  2022-03-20 09:00:00+0800                     15.000000
241   checkoutservice-1                    trace.from_root.duration_p95    0.003441                           27                                2  2022-03-20 09:00:00+0800                      0.003617
437  checkoutservice2-0         trace.to_paymentservice2-0.duration_p95    0.000158                           30                                2  2022-03-20 09:00:00+0800                      0.000160
624          frontend-0  trace.to_productcatalogservice-0.duration_mean    0.004077                           30                                2  2022-03-20 09:00:00+0800                      0.004168
752          frontend-2                   trace.from_root.duration_mean    0.043418                           30                                2  2022-03-20 09:00:00+0800                      0.043683
..                  ...                                             ...         ...                          ...                              ...                       ...                           ...
583          frontend-0                  trace.to_adservice-1.row_count   10.100000                           30                                2  2022-03-20 09:01:00+0800                     11.000000
584          frontend-0              trace.to_adservice-2.duration_mean    0.000021                           30                                2  2022-03-20 09:01:00+0800                      0.000034
587          frontend-0                  trace.to_adservice-2.row_count   10.100000                           30                                2  2022-03-20 09:01:00+0800                     11.000000
628          frontend-0  trace.to_productcatalogservice-1.duration_mean    0.004040                           30                                2  2022-03-20 09:01:00+0800                      0.004172
629          frontend-0   trace.to_productcatalogservice-1.duration_p95    0.005403                           30                                2  2022-03-20 09:01:00+0800                      0.005803

[50 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse metric_df and tz if present, else load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using full series (no time filtering)
grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
global_p95 = grp.quantile(0.95)
global_p05 = grp.quantile(0.05)
global_counts = grp.size()

thresholds_df = (
    pd.DataFrame({
        'global_p95': global_p95,
        'global_p05': global_p05,
        'total_points_in_full_series': global_counts
    })
    .reset_index()
)

# 2) Filter metric records to incident window [2022-03-20 09:00:00, 2022-03-20 09:30:00) UTC+8
start_dt = tz.localize(pd.Timestamp('2022-03-20 09:00:00'))
end_dt = tz.localize(pd.Timestamp('2022-03-20 09:30:00'))  # exclusive upper bound

metric_incident = metric_df.copy()
metric_incident['dt'] = pd.to_datetime(metric_incident['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
metric_incident = metric_incident[(metric_incident['dt'] >= start_dt) & (metric_incident['dt'] < end_dt)].copy()

# 3) Join thresholds onto incident rows and mark anomalies (value > P95 OR value < P05)
metric_incident = metric_incident.merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')
metric_incident['is_anomaly'] = (metric_incident['value'] > metric_incident['global_p95']) | (metric_incident['value'] < metric_incident['global_p05'])
# Keep dt as timezone-aware; also prepare dt_str for compact display when needed
metric_incident['dt_str'] = metric_incident['dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 4) Extract anomalies only
anomalies_df = metric_incident[metric_incident['is_anomaly']].copy()

# 5) Aggregate anomalies by cmdb_id
if anomalies_df.shape[0] > 0:
    # total anomaly count per cmdb_id
    agg_total = anomalies_df.groupby('cmdb_id').size().rename('total_anomaly_count').reset_index()
    # distinct KPI count with anomalies
    agg_kpi_distinct = anomalies_df.groupby('cmdb_id')['kpi_name'].nunique().rename('distinct_kpi_count_with_anomalies').reset_index()
    # earliest and latest anomaly timestamps per cmdb_id
    agg_earliest = anomalies_df.groupby('cmdb_id')['dt'].min().rename('earliest_anomaly_dt').reset_index()
    agg_latest = anomalies_df.groupby('cmdb_id')['dt'].max().rename('latest_anomaly_dt').reset_index()
    # top 3 kpis by anomaly count per cmdb_id
    kpi_counts = anomalies_df.groupby(['cmdb_id', 'kpi_name']).size().rename('kpi_anomaly_count').reset_index()
    # build a helper that aggregates top-3 kpis into a comma-separated string like "kpiA(5),kpiB(3),kpiC(2)"
    def top_kpis(series_df):
        top = series_df.sort_values('kpi_anomaly_count', ascending=False).head(3)
        return ",".join([f"{r.kpi_name}({int(r.kpi_anomaly_count)})" for _, r in top.iterrows()])
    top_kpis_by_cmdb = kpi_counts.groupby('cmdb_id').apply(top_kpis).rename('top_3_kpis_by_anomaly_count').reset_index()
    
    # Merge all aggregates
    anomalies_by_cmdb = agg_total.merge(agg_kpi_distinct, on='cmdb_id')\
                                .merge(agg_earliest, on='cmdb_id')\
                                .merge(agg_latest, on='cmdb_id')\
                                .merge(top_kpis_by_cmdb, on='cmdb_id')
else:
    # empty structure with requested columns
    anomalies_by_cmdb = pd.DataFrame(columns=[
        'cmdb_id',
        'total_anomaly_count',
        'distinct_kpi_count_with_anomalies',
        'earliest_anomaly_dt',
        'latest_anomaly_dt',
        'top_3_kpis_by_anomaly_count'
    ])

# Format earliest/latest timestamps as strings, keep None for missing
def fmt_dt_col(x):
    if pd.isna(x):
        return None
    return x.strftime('%Y-%m-%d %H:%M:%S%z')

anomalies_by_cmdb['earliest_anomaly_timestamp_in_incident'] = anomalies_by_cmdb['earliest_anomaly_dt'].apply(fmt_dt_col)
anomalies_by_cmdb['latest_anomaly_timestamp_in_incident'] = anomalies_by_cmdb['latest_anomaly_dt'].apply(fmt_dt_col)

# Select and order columns as requested
final_cols = [
    'cmdb_id',
    'total_anomaly_count',
    'distinct_kpi_count_with_anomalies',
    'earliest_anomaly_timestamp_in_incident',
    'latest_anomaly_timestamp_in_incident',
    'top_3_kpis_by_anomaly_count'
]
anomalies_by_cmdb_final = anomalies_by_cmdb[final_cols].copy()

# 6) Sort by total_anomaly_count desc, then earliest timestamp asc (None last)
anomalies_by_cmdb_final['__earliest_sort'] = pd.to_datetime(anomalies_by_cmdb_final['earliest_anomaly_timestamp_in_incident'], format='%Y-%m-%d %H:%M:%S%z', utc=True, errors='coerce')
anomalies_by_cmdb_final = anomalies_by_cmdb_final.sort_values(by=['total_anomaly_count', '__earliest_sort'], ascending=[False, True], na_position='last')
anomalies_by_cmdb_final = anomalies_by_cmdb_final.drop(columns='__earliest_sort')

# Limit to top 20
anomalies_by_cmdb_top20 = anomalies_by_cmdb_final.head(20)

# Store intermediates for later steps
thresholds_df, metric_incident, anomalies_df, kpi_counts, anomalies_by_cmdb, anomalies_by_cmdb_top20

```
Out[4]:
```
Summary — system-wide metric anomalies (incident window 2022-03-20 09:00–09:30 UTC+8)

- Method: global P95 and P05 were computed per (cmdb_id, kpi_name) using the full available series. Metric records inside the incident window were flagged as anomalies when value > global_P95 or value < global_P05.

- Overall counts: 5,481 anomaly records were detected in the incident window, affecting ~59 distinct cmdb_id entities.

- Top affected components (top 20 shown; sorted by anomaly count):
  - frontend-1: 257 anomalies (73 distinct KPIs)
  - frontend-0: 256 anomalies (73 distinct KPIs)
  - frontend2-0: 245 anomalies (71 distinct KPIs)
  - frontend-2: 236 anomalies (70 distinct KPIs)
  - checkoutservice-1: 181 anomalies (63 distinct KPIs)
  - checkoutservice2-0: 149 anomalies (61 distinct KPIs)
  - adservice / adservice2: each ~140 anomalies (41 distinct KPIs)
  - several productcatalogservice and cartservice instances and nodes also appear in the top-20 (counts in the 110–160 range)
  - nodes such as node-6, node-1, node-2, node-3 show elevated anomaly counts (e.g., node-6: 130 anomalies)

- Time coverage of anomalies: earliest detected anomaly timestamps are at 2022-03-20 09:00:00+08:00 and latest at 2022-03-20 09:29:00+08:00 for the top components — anomalies are distributed across the entire 30-minute window.

- Typical KPIs with most anomalies (examples from top_kpis_by_anomaly_count):
  - Frontend pods: container CPU metrics (e.g., container_cpu_cfs_periods, container_cpu_usage_seconds) are among the top anomalous KPIs.
  - adservice/adservice2: app.grpc.* metrics and runtime Java GC metrics appear frequently in their top anomalies.
  - Nodes: system CPU metrics (pct_usage, user, iowait) and disk/fs metrics show up as top anomalous KPIs for node entries.

- Interpretation: anomalies are widespread across frontends, several services (adservice, checkoutservice, productcatalogservice, cartservice) and several nodes. Many components have multiple KPIs with anomalies, dominated by container CPU metrics and some service-specific metrics (gRPC, GC). The anomaly pattern suggests broad, short-lived deviations across containers and nodes during the incident window rather than a single isolated KPI.

If you want, I can:
- return the compact top-20 table shown,
- list all cmdb_ids with >N anomalies,
- or drill into anomalies for a specific component or KPI (timestamps + values).

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name    global_p95    global_p05  total_points_in_full_series
0              adservice                                     app.grpc.count    102.650000     66.000000                           30
1              adservice                                       app.grpc.mrt      0.002348      0.002170                           30
2              adservice                                        app.grpc.rr    100.000000    100.000000                           30
3              adservice                                        app.grpc.sr     50.357143     42.155585                           30
4              adservice                                     app.http.count     24.000000     24.000000                           30
...                  ...                                                ...           ...           ...                          ...
4725  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     10.050000      5.725000                           30
4726  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  33200.000000  17745.000000                           30
4727  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.001025      0.000000                           30
4728  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     11.500000      6.500000                           30
4729  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....   1144.250000    641.775000                           30

[4730 rows x 5 columns],          timestamp             cmdb_id                                           kpi_name         value                        dt    global_p95    global_p05  total_points_in_full_series  is_anomaly                    dt_str
0       1647738000           adservice                                     app.grpc.count     72.000000 2022-03-20 09:00:00+08:00    102.650000     66.000000                           30       False  2022-03-20 09:00:00+0800
1       1647738000           adservice                                       app.grpc.mrt      0.002187 2022-03-20 09:00:00+08:00      0.002348      0.002170                           30       False  2022-03-20 09:00:00+0800
2       1647738000           adservice                                        app.grpc.rr    100.000000 2022-03-20 09:00:00+08:00    100.000000    100.000000                           30       False  2022-03-20 09:00:00+0800
3       1647738000           adservice                                        app.grpc.sr     47.222222 2022-03-20 09:00:00+08:00     50.357143     42.155585                           30       False  2022-03-20 09:00:00+0800
4       1647738000           adservice                                     app.http.count     24.000000 2022-03-20 09:00:00+08:00     24.000000     24.000000                           30       False  2022-03-20 09:00:00+0800
...            ...                 ...                                                ...           ...                       ...           ...           ...                          ...         ...                       ...
141607  1647739740  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...      8.000000 2022-03-20 09:29:00+08:00     10.050000      5.725000                           30       False  2022-03-20 09:29:00+0800
141608  1647739740  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  26550.000000 2022-03-20 09:29:00+08:00  33200.000000  17745.000000                           30       False  2022-03-20 09:29:00+0800
141609  1647739740  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.000000 2022-03-20 09:29:00+08:00      0.001025      0.000000                           30       False  2022-03-20 09:29:00+0800
141610  1647739740  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      9.500000 2022-03-20 09:29:00+08:00     11.500000      6.500000                           30       False  2022-03-20 09:29:00+0800
141611  1647739740  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....    895.500000 2022-03-20 09:29:00+08:00   1144.250000    641.775000                           30       False  2022-03-20 09:29:00+0800

[141612 rows x 10 columns],          timestamp             cmdb_id                                           kpi_name         value                        dt    global_p95    global_p05  total_points_in_full_series  is_anomaly                    dt_str
14      1647738000           adservice  runtime.java_lang_GarbageCollector_CollectionC...  3.182208e+04 2022-03-20 09:00:00+08:00  3.205717e+04  3.183445e+04                           30        True  2022-03-20 09:00:00+0800
16      1647738000           adservice  runtime.java_lang_GarbageCollector_CollectionT...  5.004933e+04 2022-03-20 09:00:00+08:00  5.041756e+04  5.006881e+04                           30        True  2022-03-20 09:00:00+0800
20      1647738000           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.000000e+00 2022-03-20 09:00:00+08:00  1.916667e+00  1.287500e+00                           30        True  2022-03-20 09:00:00+0800
22      1647738000           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.149451e+08 2022-03-20 09:00:00+08:00  2.165981e+08  2.150323e+08                           30        True  2022-03-20 09:00:00+0800
24      1647738000           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  3.182208e+04 2022-03-20 09:00:00+08:00  3.205717e+04  3.183445e+04                           30        True  2022-03-20 09:00:00+0800
...            ...                 ...                                                ...           ...                       ...           ...           ...                          ...         ...                       ...
141553  1647739740  shippingservice2-0   container.node-5.container_fs_usage_MB./dev/vda1  2.653125e+01 2022-03-20 09:29:00+08:00  2.652559e+01  2.642578e+01                           30        True  2022-03-20 09:29:00+0800
141559  1647739740  shippingservice2-0               container.node-5.container_last_seen  1.647740e+09 2022-03-20 09:29:00+08:00  1.647740e+09  1.647738e+09                           30        True  2022-03-20 09:29:00+0800
141568  1647739740  shippingservice2-0              container.node-5.container_memory_rss  1.488896e+07 2022-03-20 09:29:00+08:00  1.486029e+07  1.448325e+07                           30        True  2022-03-20 09:29:00+0800
141570  1647739740  shippingservice2-0         container.node-5.container_memory_usage_MB  2.824609e+01 2022-03-20 09:29:00+08:00  2.821875e+01  2.785918e+01                           30        True  2022-03-20 09:29:00+0800
141571  1647739740  shippingservice2-0   container.node-5.container_memory_working_set_MB  1.911328e+01 2022-03-20 09:29:00+08:00  1.908594e+01  1.872637e+01                           30        True  2022-03-20 09:29:00+0800

[5481 rows x 10 columns],                  cmdb_id                                           kpi_name  kpi_anomaly_count
0              adservice                                     app.grpc.count                  3
1              adservice                                       app.grpc.mrt                  4
2              adservice                                        app.grpc.sr                  4
3              adservice                                       app.http.mrt                  4
4              adservice  runtime.java_lang_GarbageCollector_CollectionC...                  4
...                  ...                                                ...                ...
1722  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...                  4
1723  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                  3
1724  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                  2
1725  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                  2
1726  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                  3

[1727 rows x 3 columns],                cmdb_id  total_anomaly_count  distinct_kpi_count_with_anomalies       earliest_anomaly_dt         latest_anomaly_dt                        top_3_kpis_by_anomaly_count earliest_anomaly_timestamp_in_incident latest_anomaly_timestamp_in_incident
0            adservice                  140                                 41 2022-03-20 09:00:00+08:00 2022-03-20 09:29:00+08:00  app.grpc.mrt(4),runtime.java_lang_GarbageColle...               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800
1          adservice-0                   76                                 24 2022-03-20 09:00:00+08:00 2022-03-20 09:29:00+08:00  container.node-5.container_cpu_cfs_periods(4),...               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800
2          adservice-1                   76                                 21 2022-03-20 09:00:00+08:00 2022-03-20 09:29:00+08:00  container.node-5.container_cpu_cfs_periods(4),...               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800
3          adservice-2                   88                                 25 2022-03-20 09:00:00+08:00 2022-03-20 09:29:00+08:00  container.node-5.container_cpu_cfs_periods(4),...               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800
4           adservice2                  140                                 41 2022-03-20 09:00:00+08:00 2022-03-20 09:29:00+08:00  runtime.java_lang_GarbageCollector_CollectionC...               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800
..                 ...                  ...                                ...                       ...                       ...                                                ...                                    ...                                  ...
54     shippingservice                    5                                  2 2022-03-20 09:08:00+08:00 2022-03-20 09:28:00+08:00                  app.grpc.count(3),app.grpc.mrt(2)               2022-03-20 09:08:00+0800             2022-03-20 09:28:00+0800
55   shippingservice-0                   85                                 29 2022-03-20 09:00:00+08:00 2022-03-20 09:29:00+08:00  container.node-5.container_cpu_usage_seconds(4...               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800
56   shippingservice-1                   81                                 29 2022-03-20 09:00:00+08:00 2022-03-20 09:29:00+08:00  container.node-5.container_cpu_usage_seconds(4...               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800
57   shippingservice-2                   85                                 29 2022-03-20 09:00:00+08:00 2022-03-20 09:29:00+08:00  container.node-5.container_cpu_usage_seconds(4...               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800
58  shippingservice2-0                   76                                 26 2022-03-20 09:00:00+08:00 2022-03-20 09:29:00+08:00  container.node-5.container_cpu_usage_seconds(4...               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800

[59 rows x 8 columns],                      cmdb_id  total_anomaly_count  distinct_kpi_count_with_anomalies earliest_anomaly_timestamp_in_incident latest_anomaly_timestamp_in_incident                        top_3_kpis_by_anomaly_count
28                frontend-1                  257                                 73               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-5.container_cpu_cfs_periods(4),...
27                frontend-0                  256                                 73               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-5.container_cpu_usage_seconds(4...
30               frontend2-0                  245                                 71               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-6.container_cpu_system_seconds(...
29                frontend-2                  236                                 70               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-5.container_cpu_cfs_periods(4),...
13         checkoutservice-1                  181                                 63               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-5.container_cpu_usage_seconds(4...
44   productcatalogservice-1                  158                                 45               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-5.container_cpu_cfs_periods(4),...
15        checkoutservice2-0                  149                                 61               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-5.container_cpu_usage_seconds(4...
45   productcatalogservice-2                  146                                 46               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-5.container_cpu_cfs_periods(4),...
43   productcatalogservice-0                  141                                 46               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-5.container_cpu_cfs_periods(4),...
0                  adservice                  140                                 41               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  app.grpc.mrt(4),runtime.java_lang_GarbageColle...
4                 adservice2                  140                                 41               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  runtime.java_lang_GarbageCollector_CollectionC...
14         checkoutservice-2                  137                                 62               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-5.container_cpu_cfs_periods(4),...
36                    node-6                  130                                 37               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  system.cpu.pct_usage(4),system.cpu.user(4),sys...
46  productcatalogservice2-0                  126                                 38               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-6.container_cpu_cfs_periods(4),...
31                    node-1                  124                                 36               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  system.cpu.iowait(4),system.cpu.pct_usage(4),s...
32                    node-2                  119                                 35               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  system.cpu.iowait(4),system.cpu.pct_usage(4),s...
10            cartservice2-0                  116                                 40               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-6.container_cpu_cfs_periods(4),...
8              cartservice-1                  111                                 31               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-5.container_cpu_cfs_periods(4),...
33                    node-3                  111                                 35               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  system.cpu.pct_usage(4),system.disk.free(4),sy...
20        currencyservice2-0                  107                                 31               2022-03-20 09:00:00+0800             2022-03-20 09:29:00+0800  container.node-6.container_fs_usage_MB./dev/vd...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse anomalies_df if present; otherwise reconstruct from previous state
try:
    anomalies_df
except NameError:
    # Reconstruct anomalies_df from metric_incident if needed
    try:
        metric_incident
    except NameError:
        metric_df = pd.read_csv("metric.csv")
        tz = pytz.timezone('Asia/Shanghai')
        metric_incident = metric_df.copy()
        metric_incident['dt'] = pd.to_datetime(metric_incident['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        start_dt = tz.localize(pd.Timestamp('2022-03-20 09:00:00'))
        end_dt = tz.localize(pd.Timestamp('2022-03-20 09:30:00'))
        metric_incident = metric_incident[(metric_incident['dt'] >= start_dt) & (metric_incident['dt'] < end_dt)].copy()
        grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
        thresholds_df = pd.DataFrame({
            'global_p95': grp.quantile(0.95),
            'global_p05': grp.quantile(0.05),
            'total_points_in_full_series': grp.size()
        }).reset_index()
        metric_incident = metric_incident.merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')
        metric_incident['is_anomaly'] = (metric_incident['value'] > metric_incident['global_p95']) | (metric_incident['value'] < metric_incident['global_p05'])
    anomalies_df = metric_incident[metric_incident['is_anomaly']].copy()

# Ensure timezone
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure dt column exists and is timezone-aware
if 'dt' not in anomalies_df.columns:
    anomalies_df['dt'] = pd.to_datetime(anomalies_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    anomalies_df['dt'] = pd.to_datetime(anomalies_df['dt']).dt.tz_convert(tz)

# Sort anomalies
anomalies_sorted = anomalies_df.sort_values(['cmdb_id', 'kpi_name', 'dt']).copy()

# Compute gap (minutes) to previous anomaly within each (cmdb_id, kpi_name)
anomalies_sorted['prev_dt'] = anomalies_sorted.groupby(['cmdb_id', 'kpi_name'])['dt'].shift(1)
anomalies_sorted['diff_min'] = (anomalies_sorted['dt'] - anomalies_sorted['prev_dt']).dt.total_seconds() / 60.0
anomalies_sorted['new_run'] = (anomalies_sorted['diff_min'].isna()) | (anomalies_sorted['diff_min'] > 1.0)
anomalies_sorted['run_id'] = anomalies_sorted.groupby(['cmdb_id', 'kpi_name'])['new_run'].cumsum()

# Aggregate basic run-level stats
group_keys = ['cmdb_id', 'kpi_name', 'run_id']
run_basic = anomalies_sorted.groupby(group_keys).agg(
    run_start_dt = ('dt', 'min'),
    run_end_dt = ('dt', 'max'),
    run_length = ('dt', 'count'),
    max_value_in_run = ('value', 'max'),
    min_value_in_run = ('value', 'min'),
    global_p95 = ('global_p95', 'first'),
    global_p05 = ('global_p05', 'first')
).reset_index()

# Compute count_high and count_low per run
count_high = anomalies_sorted.assign(is_high=anomalies_sorted['value'] > anomalies_sorted['global_p95']).groupby(group_keys)['is_high'].sum().rename('count_high').reset_index()
count_low = anomalies_sorted.assign(is_low=anomalies_sorted['value'] < anomalies_sorted['global_p05']).groupby(group_keys)['is_low'].sum().rename('count_low').reset_index()

# Merge counts into run_basic
run_agg = run_basic.merge(count_high, on=group_keys, how='left').merge(count_low, on=group_keys, how='left')
run_agg['count_high'] = run_agg['count_high'].fillna(0).astype(int)
run_agg['count_low'] = run_agg['count_low'].fillna(0).astype(int)

# Decide direction and compute severity_percent
def decide_direction(row):
    return 'high' if row['count_high'] >= row['count_low'] else 'low'

run_agg['anomaly_direction'] = run_agg.apply(decide_direction, axis=1)

eps = 1e-9
def compute_severity(row):
    if row['anomaly_direction'] == 'high':
        denom = max(eps, abs(row['global_p95']))
        return (row['max_value_in_run'] - row['global_p95']) / denom * 100.0
    else:
        denom = max(eps, abs(row['global_p05']))
        return (row['global_p05'] - row['min_value_in_run']) / denom * 100.0

run_agg['severity_percent'] = run_agg.apply(compute_severity, axis=1)

# Keep runs with run_length >= 2
runs_filtered = run_agg[run_agg['run_length'] >= 2].copy()

# Format timestamps as strings (UTC+8)
runs_filtered['run_start_timestamp'] = runs_filtered['run_start_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
runs_filtered['run_end_timestamp'] = runs_filtered['run_end_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Select requested compact columns
final_cols = [
    'cmdb_id',
    'kpi_name',
    'run_start_timestamp',
    'run_end_timestamp',
    'run_length',
    'anomaly_direction',
    'max_value_in_run',
    'min_value_in_run',
    'global_p95',
    'global_p05',
    'severity_percent'
]
run_df_all = runs_filtered[final_cols].copy()

# Sort by severity_percent desc then run_start asc
run_df_all['__start_sort'] = pd.to_datetime(run_df_all['run_start_timestamp'], format='%Y-%m-%d %H:%M:%S%z', utc=True, errors='coerce')
run_df_all = run_df_all.sort_values(by=['severity_percent', '__start_sort'], ascending=[False, True], na_position='last').drop(columns='__start_sort')

# Top-20 compact output
run_top20 = run_df_all.head(20)

# Return only the compact top-20 table (store full run_df_all for later use)
run_df_all, run_top20

```
Out[5]:
```
Summary — consecutive anomaly runs (runs of >=2 consecutive anomalous minutes) detected in metric data during 2022-03-20 09:00–09:30 (UTC+8)

- Scope & method: Using global P95/P05 computed from full series, we flagged incident-window metric points as anomalies and grouped consecutive anomaly points per (cmdb_id, kpi_name). We kept runs with length >= 2 minutes.

- Counts: 872 runs met the run_length >= 2 criterion across the system during the 30‑minute window.

- Severity ranking (top examples from detected runs, sorted by severity_percent):
  - emailservice-2 — container.node-5.container_network_receive_MB...: 2-minute high run (09:19–09:20), max=0.56105, global_p95≈0.027317, severity ≈ 1953% over P95.
  - shippingservice-0 — same network_receive_MB KPI: 2-minute high run (09:27–09:28), severity ≈ 1612%.
  - cartservice-1 — same network_receive_MB KPI: 2-minute high run (09:20–09:21), severity ≈ 1073%.
  - node-1 — system.cpu.iowait: 2-minute high run (09:00–09:01), max=6.64, global_p95≈1.318, severity ≈ 404%.
  - node-6 — system.io.await and system.io.w_await: 2-minute high runs around 09:25–09:26, severity ≈ 293%.
  - frontend-0 — container network_receive_MB: 2-minute high run (09:21–09:22), severity ≈ 284%.
  - cartservice2-0 — mesh.destination.frontend2.cartservice2.istio_*: a 4-minute high run (09:20–09:23), severity ≈ 162%.
  - node-6 — large increases in net bytes_sent (09:01–09:02), severity ≈ 163%.
  - Several runs flagged as "low" (value < global_P05) appear — e.g., recommendationservice-1 container_cpu_system_seconds had a 3-minute low run (09:13–09:15) with severity effectively 0% relative to tiny baseline (computed but small due to tiny denominator).

- Pattern summary:
  - Many of the highest-severity runs involve network throughput metrics (container network receive/transmit), node network/tcp counters, and node CPU/io metrics.
  - Most runs are short (2–4 minutes) but with very large deviations relative to their global P95 for some network/IO KPIs.
  - Both "high" (value > P95) and some "low" runs are present; high-direction runs dominate the top severity list.

- Practical next steps:
  - Inspect the top-ranked runs (start timestamps and KPIs above) with logs/traces at the same minutes to find correlation (e.g., packet bursts, GC pauses, or request spikes).
  - Correlate these runs with error_logs and trace spikes (earlier analysis showed frontend warnings concentrated in the same window).
  - If you want, I can return the compact top-20 runs table, full run table, or filter runs for a specific component/KPI and show the underlying incident-window points.

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                                           kpi_name       run_start_timestamp         run_end_timestamp  run_length anomaly_direction  max_value_in_run  min_value_in_run    global_p95    global_p05  severity_percent
1805           emailservice-2  container.node-5.container_network_receive_MB....  2022-03-20 09:19:00+0800  2022-03-20 09:20:00+0800           2              high      5.610495e-01      2.784872e-02  2.731700e-02  1.790259e-02      1.953847e+03
4310        shippingservice-0  container.node-5.container_network_receive_MB....  2022-03-20 09:27:00+0800  2022-03-20 09:28:00+0800           2              high      5.559916e-01      3.280449e-02  3.247747e-02  2.321289e-02      1.611930e+03
601             cartservice-1  container.node-5.container_network_receive_MB....  2022-03-20 09:20:00+0800  2022-03-20 09:21:00+0800           2              high      5.702920e-01      3.415203e-02  4.860063e-02  3.482947e-02      1.073425e+03
2714                   node-1                                  system.cpu.iowait  2022-03-20 09:00:00+0800  2022-03-20 09:01:00+0800           2              high      6.640000e+00      3.800000e-01  1.318000e+00  4.490000e-01      4.037936e+02
3193                   node-6                                    system.io.await  2022-03-20 09:25:00+0800  2022-03-20 09:26:00+0800           2              high      2.170000e+00      0.000000e+00  5.520000e-01  6.300000e-02      2.931159e+02
...                       ...                                                ...                       ...                       ...         ...               ...               ...               ...           ...           ...               ...
4422        shippingservice-2               container.node-5.container_last_seen  2022-03-20 09:28:00+0800  2022-03-20 09:29:00+0800           2              high      1.647740e+09      1.647740e+09  1.647740e+09  1.647738e+09      5.279960e-06
4487       shippingservice2-0               container.node-5.container_last_seen  2022-03-20 09:28:00+0800  2022-03-20 09:29:00+0800           2              high      1.647740e+09      1.647740e+09  1.647740e+09  1.647738e+09      5.279960e-06
4019  recommendationservice-1      container.node-5.container_cpu_system_seconds  2022-03-20 09:13:00+0800  2022-03-20 09:15:00+0800           3               low      1.500000e-02      5.000000e-03  1.500000e-02  5.000000e-03      1.277971e-10
3663  productcatalogservice-1  mesh.destination.recommendationservice.product...  2022-03-20 09:00:00+0800  2022-03-20 09:01:00+0800           2              high      5.250000e-04      5.250000e-04  5.250000e-04  0.000000e+00      1.714072e-11
579             cartservice-1        container.node-5.container_cpu_user_seconds  2022-03-20 09:19:00+0800  2022-03-20 09:20:00+0800           2              high      1.100000e-01      7.500000e-02  1.100000e-01  7.950000e-02      1.425627e-12

[872 rows x 11 columns],                        cmdb_id                                           kpi_name       run_start_timestamp         run_end_timestamp  run_length anomaly_direction  max_value_in_run  min_value_in_run    global_p95     global_p05  severity_percent
1805            emailservice-2  container.node-5.container_network_receive_MB....  2022-03-20 09:19:00+0800  2022-03-20 09:20:00+0800           2              high      5.610495e-01      2.784872e-02  2.731700e-02       0.017903       1953.847322
4310         shippingservice-0  container.node-5.container_network_receive_MB....  2022-03-20 09:27:00+0800  2022-03-20 09:28:00+0800           2              high      5.559916e-01      3.280449e-02  3.247747e-02       0.023213       1611.930078
601              cartservice-1  container.node-5.container_network_receive_MB....  2022-03-20 09:20:00+0800  2022-03-20 09:21:00+0800           2              high      5.702920e-01      3.415203e-02  4.860063e-02       0.034829       1073.425207
2714                    node-1                                  system.cpu.iowait  2022-03-20 09:00:00+0800  2022-03-20 09:01:00+0800           2              high      6.640000e+00      3.800000e-01  1.318000e+00       0.449000        403.793627
3193                    node-6                                    system.io.await  2022-03-20 09:25:00+0800  2022-03-20 09:26:00+0800           2              high      2.170000e+00      0.000000e+00  5.520000e-01       0.063000        293.115942
3210                    node-6                                  system.io.w_await  2022-03-20 09:25:00+0800  2022-03-20 09:26:00+0800           2              high      2.170000e+00      0.000000e+00  5.520000e-01       0.063000        293.115942
1905                frontend-0  container.node-5.container_network_receive_MB....  2022-03-20 09:21:00+0800  2022-03-20 09:22:00+0800           2              high      6.826787e-01      1.039734e-01  1.779889e-01       0.132615        283.551254
2595               frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...  2022-03-20 09:20:00+0800  2022-03-20 09:21:00+0800           2              high      4.887000e-01      1.891000e-01  1.472050e-01       0.056661        231.986006
3240                    node-6                              system.net.bytes_sent  2022-03-20 09:01:00+0800  2022-03-20 09:02:00+0800           2              high      5.929700e+05      2.614937e+05  2.254294e+05   42252.862500        163.040189
816             cartservice2-0  mesh.destination.frontend2.cartservice2.istio_...  2022-03-20 09:20:00+0800  2022-03-20 09:23:00+0800           4              high      3.981250e-01      3.040000e-02  1.518125e-01       0.030948        162.247839
3853  productcatalogservice2-0  container.node-6.container_cpu_cfs_throttled_s...  2022-03-20 09:14:00+0800  2022-03-20 09:15:00+0800           2              high      2.897740e-01      1.487590e-01  1.238994e-01       0.000000        133.878493
764             cartservice2-0   container.node-6.container_fs_writes_MB./dev/vda  2022-03-20 09:16:00+0800  2022-03-20 09:17:00+0800           2              high      1.289062e-01      1.015625e-01  5.585937e-02       0.000000        130.769231
2788                    node-1                              system.net.bytes_sent  2022-03-20 09:08:00+0800  2022-03-20 09:09:00+0800           2              high      7.385169e+06      4.002420e+06  3.202465e+06  723831.754500        130.608894
2800                    node-1                            system.net.tcp.out_segs  2022-03-20 09:08:00+0800  2022-03-20 09:09:00+0800           2              high      3.590421e+04      2.029362e+04  1.596727e+04    3530.645500        124.861287
763             cartservice2-0      container.node-6.container_fs_writes./dev/vda  2022-03-20 09:16:00+0800  2022-03-20 09:17:00+0800           2              high      1.800000e+01      1.500000e+01  8.250000e+00       0.000000        118.181818
2724                    node-1                                    system.cpu.user  2022-03-20 09:00:00+0800  2022-03-20 09:01:00+0800           2              high      3.705000e+01      2.767000e+01  1.749550e+01       2.079000        111.768741
493                cartservice                                       app.grpc.mrt  2022-03-20 09:20:00+0800  2022-03-20 09:21:00+0800           2              high      4.703604e-03      2.544737e-03  2.284435e-03       0.001021        105.897913
1772            emailservice-1  mesh.destination.checkoutservice.emailservice....  2022-03-20 09:04:00+0800  2022-03-20 09:06:00+0800           3               low      2.162500e+03      0.000000e+00  1.945875e+03     185.625000        100.000000
1774            emailservice-1  mesh.destination.checkoutservice.emailservice....  2022-03-20 09:04:00+0800  2022-03-20 09:06:00+0800           3               low      2.625000e-03      0.000000e+00  2.388750e-03       0.000236        100.000000
1780            emailservice-1  mesh.destination.checkoutservice.emailservice....  2022-03-20 09:04:00+0800  2022-03-20 09:06:00+0800           3               low      3.125000e+03      0.000000e+00  2.843750e+03     281.250000        100.000000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse existing trace_df and tz if available; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and total points per (cmdb_id, trace_name) using full trace.csv
grp_full = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_global_p95 = grp_full.quantile(0.95)
trace_total_points = grp_full.size()

df_trace_global = (
    pd.DataFrame({
        'global_p95': trace_global_p95,
        'total_points_in_full_series': trace_total_points
    })
    .reset_index()
)

# 2) Filter trace records to frontends and incident subwindow 09:00:00 - 09:05:00 (UTC+8)
frontend_ids = ['frontend-0', 'frontend-1', 'frontend-2']
start_dt = tz.localize(pd.Timestamp('2022-03-20 09:00:00'))
end_dt = tz.localize(pd.Timestamp('2022-03-20 09:05:00'))  # exclusive upper bound

# create dt column
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 3) Restrict trace_name to those containing specified keywords
keywords = ['error_rate', 'duration_p95', 'duration_mean', 'row_count']
pattern = '|'.join(keywords)

# Prepare list of (cmdb_id, trace_name) pairs of interest from global table (ensures global thresholds computed before filtering)
df_pairs = df_trace_global[
    (df_trace_global['cmdb_id'].isin(frontend_ids)) &
    (df_trace_global['trace_name'].str.contains(pattern, regex=True))
].copy()

# Filter incident records to frontends, time window, and keyword-trace_names
trace_incident = trace_df[
    (trace_df['cmdb_id'].isin(frontend_ids)) &
    (trace_df['dt'] >= start_dt) &
    (trace_df['dt'] < end_dt) &
    (trace_df['trace_name'].str.contains(pattern, regex=True))
].copy()

# 4) Join global thresholds to incident rows (we computed thresholds globally above)
trace_incident_join = trace_incident.merge(df_pairs, on=['cmdb_id', 'trace_name'], how='inner')

# Mark above-P95 (only > global_p95 as requested)
trace_incident_join['above_p95'] = trace_incident_join['value'] > trace_incident_join['global_p95']
trace_incident_join['dt_above'] = trace_incident_join['dt'].where(trace_incident_join['above_p95'], pd.NaT)

# Aggregate incident stats per (cmdb_id, trace_name)
trace_incident_agg = (
    trace_incident_join
    .groupby(['cmdb_id', 'trace_name'])
    .agg(
        incident_points_count_above_p95 = ('above_p95', 'sum'),
        earliest_timestamp_above = ('dt_above', 'min'),
        max_value_in_incident_window = ('value', 'max'),
        incident_points_total = ('value', 'count')
    )
    .reset_index()
)

# Ensure pairs with zero incident rows are included by left-joining aggregation onto df_pairs
trace_result = df_pairs.merge(trace_incident_agg, on=['cmdb_id', 'trace_name'], how='left')

# Fill missing numeric counts with 0
trace_result['incident_points_count_above_p95'] = trace_result['incident_points_count_above_p95'].fillna(0).astype(int)
trace_result['incident_points_total'] = trace_result['incident_points_total'].fillna(0).astype(int)
trace_result['max_value_in_incident_window'] = trace_result['max_value_in_incident_window']  # may be NaN if no incident rows

# Format earliest timestamp as string or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return x.strftime('%Y-%m-%d %H:%M:%S%z')

trace_result['earliest_timestamp_above'] = trace_result['earliest_timestamp_above'].apply(fmt_ts)

# Select required columns and sort by incident_points_count_above_p95 desc then earliest timestamp asc
final_cols = [
    'cmdb_id',
    'trace_name',
    'global_p95',
    'total_points_in_full_series',
    'incident_points_count_above_p95',
    'earliest_timestamp_above',
    'max_value_in_incident_window',
    'incident_points_total'
]
trace_final = trace_result[final_cols].copy()

# Sorting: convert earliest timestamp back to datetime for correct ordering (None last)
trace_final['__earliest_sort'] = pd.to_datetime(trace_final['earliest_timestamp_above'], format='%Y-%m-%d %H:%M:%S%z', utc=True, errors='coerce')
trace_final = trace_final.sort_values(by=['incident_points_count_above_p95', '__earliest_sort'], ascending=[False, True], na_position='last')
trace_final = trace_final.drop(columns='__earliest_sort')

# Limit to top 20 rows for compact output
trace_final_top20 = trace_final.head(20)

# Store intermediates for later steps
df_trace_global, df_pairs, trace_incident, trace_incident_join, trace_incident_agg, trace_final_top20

# Display the compact top-20 table
trace_final_top20

```
Out[6]:
```
Summary — trace P95 check for frontends in 2022-03-20 09:00–09:05 (UTC+8)

- Method: global P95 computed per (cmdb_id, trace_name) using the full trace.csv. Then trace records filtered to frontend-0/1/2 and the 5‑minute subwindow 09:00–09:05. Only trace_names containing error_rate, duration_p95, duration_mean, or row_count were considered.

- Overall: most examined trace series have 5 incident points (the 5-minute window); a few series for frontend-1 had 4 points. Exceedances of the global P95 are rare and short (mostly 1–2 points per series in this window).

- Top exceedances (incident_points_count_above_p95 = 2):
  - frontend-2 — trace.to_productcatalogservice-0.duration_mean: global_p95=0.004124, 2 points > P95, earliest exceed 09:00:00, max=0.004205 (5 incident points total).
  - frontend-0 — trace.to_adservice-2.duration_mean: global_p95=0.000021, 2 points > P95, earliest exceed 09:01:00, max=0.000034 (5 incident points).
  - frontend-2 — trace.from_root.duration_p95: global_p95=0.070633, 2 points > P95, earliest exceed 09:01:00, max=0.071603 (5 incident points).
  - frontend-2 — trace.to_adservice-2.duration_mean and duration_p95, and trace.to_recommendationservice-1.duration_mean: each had 2 exceedances (earliest between 09:01–09:02) with max values slightly above P95.

- Additional notes:
  - Several row_count metrics showed single-point exceedances (e.g., to_shippingservice.*, to_cartservice.*, to_productcatalogservice.*), typically going from expected counts to +1–2 at specific minutes.
  - The exceedances are brief and limited in scope (few trace series, 1–2 minutes each), concentrated at or just after 09:00–09:02.

- Recommended next steps: inspect full traces/logs around the earliest exceedance timestamps (notably 09:00–09:02) for the listed trace targets (productcatalogservice, adservice, recommendationservice) to determine whether these short latency/row_count spikes correlate with errors or request bursts.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      trace_name  global_p95  total_points_in_full_series  incident_points_count_above_p95  earliest_timestamp_above  max_value_in_incident_window  incident_points_total
240  frontend-2  trace.to_productcatalogservice-0.duration_mean    0.004124                           30                                2  2022-03-20 09:00:00+0800                      0.004205                      5
16   frontend-0              trace.to_adservice-2.duration_mean    0.000021                           30                                2  2022-03-20 09:01:00+0800                      0.000034                      5
185  frontend-2                    trace.from_root.duration_p95    0.070633                           30                                2  2022-03-20 09:01:00+0800                      0.071603                      5
200  frontend-2              trace.to_adservice-2.duration_mean    0.000021                           30                                2  2022-03-20 09:02:00+0800                      0.000024                      5
201  frontend-2               trace.to_adservice-2.duration_p95    0.000034                           30                                2  2022-03-20 09:02:00+0800                      0.000042                      5
256  frontend-2  trace.to_recommendationservice-1.duration_mean    0.002264                           30                                2  2022-03-20 09:02:00+0800                      0.002493                      5
56   frontend-0  trace.to_productcatalogservice-0.duration_mean    0.004077                           30                                1  2022-03-20 09:00:00+0800                      0.004092                      5
87   frontend-0            trace.to_shippingservice-1.row_count    4.000000                           30                                1  2022-03-20 09:00:00+0800                      5.000000                      5
91   frontend-0            trace.to_shippingservice-2.row_count    4.000000                           30                                1  2022-03-20 09:00:00+0800                      5.000000                      5
172  frontend-1        trace.to_shippingservice-0.duration_mean    0.000070                           21                                1  2022-03-20 09:00:00+0800                      0.000090                      4
173  frontend-1         trace.to_shippingservice-0.duration_p95    0.000081                           21                                1  2022-03-20 09:00:00+0800                      0.000090                      4
184  frontend-2                   trace.from_root.duration_mean    0.043418                           30                                1  2022-03-20 09:00:00+0800                      0.043637                      5
207  frontend-2                trace.to_cartservice-0.row_count   13.550000                           30                                1  2022-03-20 09:00:00+0800                     14.000000                      5
243  frontend-2      trace.to_productcatalogservice-0.row_count   74.550000                           30                                1  2022-03-20 09:00:00+0800                     75.000000                      5
251  frontend-2      trace.to_productcatalogservice-2.row_count   74.550000                           30                                1  2022-03-20 09:00:00+0800                     75.000000                      5
255  frontend-2      trace.to_recommendationservice-0.row_count   11.550000                           30                                1  2022-03-20 09:00:00+0800                     12.000000                      5
15   frontend-0                  trace.to_adservice-1.row_count   10.100000                           30                                1  2022-03-20 09:01:00+0800                     11.000000                      5
17   frontend-0               trace.to_adservice-2.duration_p95    0.000029                           30                                1  2022-03-20 09:01:00+0800                      0.000131                      5
19   frontend-0                  trace.to_adservice-2.row_count   10.100000                           30                                1  2022-03-20 09:01:00+0800                     11.000000                      5
60   frontend-0  trace.to_productcatalogservice-1.duration_mean    0.004040                           30                                1  2022-03-20 09:01:00+0800                      0.004172                      5```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse existing variables: metric_df, run_df_all, tz
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

try:
    run_df_all
except NameError:
    # if run_df_all not present, attempt to load runs from previous 'run_df_all' variable name is required by task
    raise NameError("run_df_all not found in kernel. Please run anomaly-run detection step first.")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

frontend_ids = ['frontend-0', 'frontend-1', 'frontend-2']

# 1) Extract node tokens like 'node-<N>' from kpi_name for each frontend pod
df_frontend_metrics = metric_df[metric_df['cmdb_id'].isin(frontend_ids)].copy()
# find all occurrences of node-\d+ in kpi_name
df_frontend_metrics['node_tokens'] = df_frontend_metrics['kpi_name'].str.findall(r'node-\d+')
# explode to have one token per row
exploded = df_frontend_metrics[['cmdb_id', 'node_tokens']].explode('node_tokens').dropna(subset=['node_tokens'])
# count tokens per frontend pod
node_counts = exploded.groupby(['cmdb_id', 'node_tokens']).size().reset_index(name='count')
# sort and take top 5 per pod
node_counts = node_counts.sort_values(['cmdb_id', 'count'], ascending=[True, False])
frontend_node_map_df = node_counts.groupby('cmdb_id').head(5).reset_index(drop=True)

# 2) Filter runs table for cmdb_id in the set of node tokens and run_start between 09:00 and 09:05 UTC+8
node_tokens_set = sorted(frontend_node_map_df['node_tokens'].unique().tolist())

# convert run_df_all run_start_timestamp to timezone-aware dt for filtering
runs_dt = run_df_all.copy()
runs_dt['run_start_dt'] = pd.to_datetime(runs_dt['run_start_timestamp'], utc=True).dt.tz_convert(tz)
runs_dt['run_end_dt'] = pd.to_datetime(runs_dt['run_end_timestamp'], utc=True).dt.tz_convert(tz)

start_dt = tz.localize(pd.Timestamp('2022-03-20 09:00:00'))
end_dt = tz.localize(pd.Timestamp('2022-03-20 09:05:00'))  # exclusive

# Filter by node tokens and time window
filtered_runs = runs_dt[
    (runs_dt['cmdb_id'].isin(node_tokens_set)) &
    (runs_dt['run_start_dt'] >= start_dt) &
    (runs_dt['run_start_dt'] < end_dt)
].copy()

# Select required compact columns and sort by severity_percent desc
filtered_node_runs_df = filtered_runs[
    ['cmdb_id', 'run_start_timestamp', 'run_end_timestamp', 'run_length', 'kpi_name', 'anomaly_direction', 'severity_percent']
].sort_values(by='severity_percent', ascending=False).head(20).reset_index(drop=True)

# Display compact results (mapping and filtered runs)
frontend_node_map_df, filtered_node_runs_df

```
Out[7]:
```
Summary — frontend-to-node mapping and node-run anomalies (compact)

- Frontend -> node mapping (top node tokens per pod, up to 5):
  - frontend-0: node-5 (count 1920)
  - frontend-1: node-5 (count 1920)
  - frontend-2: node-5 (count 1920)
  => All three frontend pods predominantly reference node-5 in their metric kpi_name strings.

- Filtered consecutive anomaly runs on those node tokens within 2022-03-20 09:00:00–09:05:00 (UTC+8) — top results (sorted by severity):
  1. node-5 | system.mem.real.pct_useage | 09:00:00–09:01:00 | run_length=2 | direction=low | severity=0.327%
  2. node-5 | system.mem.pct_usage       | 09:00:00–09:01:00 | run_length=2 | direction=low | severity=0.280%
  3. node-5 | system.mem.used            | 09:00:00–09:01:00 | run_length=2 | direction=low | severity=0.275%
  4. node-5 | system.mem.free            | 09:00:00–09:01:00 | run_length=2 | direction=high| severity=0.146%
  5. node-5 | system.disk.used           | 09:00:00–09:01:00 | run_length=2 | direction=low | severity=0.004%
  6. node-5 | system.disk.free           | 09:00:00–09:01:00 | run_length=2 | direction=high| severity=0.00034%

- Interpretation:
  - All three frontend pods are associated with node-5.
  - At ~09:00–09:01, node-5 experienced short (2-minute) anomaly runs across several memory- and disk-related KPIs.
  - Memory metrics show a consistent pattern: usage/drop (low) and free/increase (high), suggesting memory was released or usage dropped briefly on node-5 at that time.
  - Severity percentages are small (<1% for these top runs), so the magnitude relative to historical P95/P05 is modest, but the co-occurrence across multiple memory/disk KPIs and frontends suggests a node-level event affecting frontends.

- Next actions (optional):
  - Inspect node-5 logs and pod lifecycle events around 09:00–09:02 for memory pressure, restarts, or garbage collection.
  - Correlate with frontend error logs/traces at 09:00–09:02 to see if user-facing errors align with these node anomalies.

I stored the frontend->node mapping and the filtered node-run table in kernel variables for further inspection if you want more rows or raw points.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id node_tokens  count
0  frontend-0      node-5   1920
1  frontend-1      node-5   1920
2  frontend-2      node-5   1920,   cmdb_id       run_start_timestamp         run_end_timestamp  run_length                    kpi_name anomaly_direction  severity_percent
0  node-5  2022-03-20 09:00:00+0800  2022-03-20 09:01:00+0800           2  system.mem.real.pct_useage               low          0.327185
1  node-5  2022-03-20 09:00:00+0800  2022-03-20 09:01:00+0800           2        system.mem.pct_usage               low          0.280158
2  node-5  2022-03-20 09:00:00+0800  2022-03-20 09:01:00+0800           2             system.mem.used               low          0.275286
3  node-5  2022-03-20 09:00:00+0800  2022-03-20 09:01:00+0800           2             system.mem.free              high          0.145880
4  node-5  2022-03-20 09:00:00+0800  2022-03-20 09:01:00+0800           2            system.disk.used               low          0.004184
5  node-5  2022-03-20 09:00:00+0800  2022-03-20 09:01:00+0800           2            system.disk.free              high          0.000338)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse existing dataframes if present, else load
try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

frontends = ['frontend-0', 'frontend-1', 'frontend-2']

# --- Part 1: error_logs summary for frontends ---
err_front = error_df[error_df['cmdb_id'].isin(frontends)].copy()
err_front['dt'] = pd.to_datetime(err_front['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# total rows and earliest/latest per cmdb_id
err_counts = err_front.groupby('cmdb_id').size().rename('total_error_rows').reset_index()
err_times = err_front.groupby('cmdb_id').agg(
    earliest_dt=('dt', 'min'),
    latest_dt=('dt', 'max')
).reset_index()
err_times['earliest_timestamp'] = err_times['earliest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
err_times['latest_timestamp'] = err_times['latest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# top 5 distinct message snippets per cmdb_id with counts and earliest sample timestamp
err_front['msg_snip'] = err_front['message'].astype(str).str.slice(0, 200)

msg_grp = (
    err_front
    .groupby(['cmdb_id', 'msg_snip'])
    .agg(count=('message', 'size'), earliest_dt=('dt', 'min'))
    .reset_index()
)
msg_grp['earliest_ts'] = msg_grp['earliest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

top_msgs_list = []
for cid, sub in msg_grp.groupby('cmdb_id'):
    top = sub.sort_values(by=['count', 'earliest_dt'], ascending=[False, True]).head(5)
    compact = "; ".join([f"{row['msg_snip']} ({int(row['count'])}) [{row['earliest_ts']}]" for _, row in top.iterrows()])
    top_msgs_list.append({'cmdb_id': cid, 'top_messages': compact})

top_msgs_df = pd.DataFrame(top_msgs_list)

# Merge into compact summary, ensure all frontends present
error_summary_df = pd.DataFrame({'cmdb_id': frontends}).merge(err_counts, on='cmdb_id', how='left') \
    .merge(err_times[['cmdb_id', 'earliest_timestamp', 'latest_timestamp']], on='cmdb_id', how='left') \
    .merge(top_msgs_df, on='cmdb_id', how='left')

error_summary_df['total_error_rows'] = error_summary_df['total_error_rows'].fillna(0).astype(int)
error_summary_df[['earliest_timestamp','latest_timestamp','top_messages']] = error_summary_df[['earliest_timestamp','latest_timestamp','top_messages']].where(pd.notna(error_summary_df[['earliest_timestamp','latest_timestamp','top_messages']]), None)

# --- Part 2: log.error_count summary for frontends in incident window ---
log_df['dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start_dt = tz.localize(pd.Timestamp('2022-03-20 09:00:00'))
end_dt = tz.localize(pd.Timestamp('2022-03-20 09:30:00'))

log_front = log_df[
    (log_df['cmdb_id'].isin(frontends)) &
    (log_df['log_name'] == 'log.error_count') &
    (log_df['dt'] >= start_dt) &
    (log_df['dt'] < end_dt)
].copy()

rows = []
for f in frontends:
    sub = log_front[log_front['cmdb_id'] == f]
    if sub.shape[0] == 0:
        rows.append({
            'cmdb_id': f,
            'sum_error_count_over_window': 0.0,
            'max_error_count_minute': 0.0,
            'earliest_minute_with_error_count_gt0': None,
            'number_of_minutes_with_error_count_gt0': 0
        })
    else:
        sum_err = float(sub['value'].sum())
        max_err = float(sub['value'].max())
        mask = sub['value'] > 0
        if mask.any():
            earliest_pos = sub.loc[mask, 'dt'].min().strftime('%Y-%m-%d %H:%M:%S%z')
            num_pos = int(mask.sum())
        else:
            earliest_pos = None
            num_pos = 0
        rows.append({
            'cmdb_id': f,
            'sum_error_count_over_window': sum_err,
            'max_error_count_minute': max_err,
            'earliest_minute_with_error_count_gt0': earliest_pos,
            'number_of_minutes_with_error_count_gt0': num_pos
        })

log_summary_df = pd.DataFrame(rows)

# Store intermediates for later use
err_front, msg_grp, top_msgs_df, log_front

# Return compact tables
error_summary_df, log_summary_df

```
Out[8]:
```
Summary — frontend error logs and log.error_count (UTC+8, incident window 2022-03-20 09:00–09:30)

Error log (error_logs.csv) per frontend:
- frontend-0: 229 error rows; earliest 2022-03-20 09:00:01+08:00; latest 2022-03-20 09:29:56+08:00. Top message snippet: "severity: warning, message: failed to retrieve..." (most frequent).
- frontend-1: 98 error rows; earliest 2022-03-20 09:00:55+08:00; latest 2022-03-20 09:29:55+08:00. Top message snippet: "severity: warning, message: failed to retrieve...".
- frontend-2: 231 error rows; earliest 2022-03-20 09:00:01+08:00; latest 2022-03-20 09:29:53+08:00. Top message snippet: "severity: warning, message: failed to retrieve...".

log.error_count (log.csv) per frontend in the same window:
- frontend-0: sum_error_count_over_window = 229.0; max per-minute = 11.0; earliest minute with errors = 2022-03-20 09:00:00+08:00; number of minutes with error_count>0 = 30.
- frontend-1: sum = 98.0; max = 7.0; earliest = 2022-03-20 09:00:00+08:00; minutes>0 = 30.
- frontend-2: sum = 231.0; max = 10.0; earliest = 2022-03-20 09:00:00+08:00; minutes>0 = 30.

Key observation:
- All three frontends show persistent errors every minute across the 30-minute window, with similar warning messages ("failed to retrieve...") and nontrivial per-minute error counts (peaks 7–11). This indicates a continuous/repeating issue affecting frontends during the incident. Recommended next steps: inspect the full error message content and backend targets referenced by "failed to retrieve..." and correlate with the metric/trace spike timestamps already identified.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_error_rows        earliest_timestamp          latest_timestamp                                       top_messages
0  frontend-0               229  2022-03-20 09:00:01+0800  2022-03-20 09:29:56+0800  severity: warning, message: failed to retrieve...
1  frontend-1                98  2022-03-20 09:00:55+0800  2022-03-20 09:29:55+0800  severity: warning, message: failed to retrieve...
2  frontend-2               231  2022-03-20 09:00:01+0800  2022-03-20 09:29:53+0800  severity: warning, message: failed to retrieve...,       cmdb_id  sum_error_count_over_window  max_error_count_minute earliest_minute_with_error_count_gt0  number_of_minutes_with_error_count_gt0
0  frontend-0                        229.0                    11.0             2022-03-20 09:00:00+0800                                      30
1  frontend-1                         98.0                     7.0             2022-03-20 09:00:00+0800                                      30
2  frontend-2                        231.0                    10.0             2022-03-20 09:00:00+0800                                      30)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# Reuse existing error_df and tz if available; otherwise load/create
try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Frontend list
frontends = ['frontend-0', 'frontend-1', 'frontend-2']

# Incident window (UTC+8)
start_dt = tz.localize(pd.Timestamp('2022-03-20 09:00:00'))
end_dt = tz.localize(pd.Timestamp('2022-03-20 09:30:00'))

# Ensure dt column timezone-aware
error_df['dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter to frontends and incident window
error_front_filtered = error_df[
    (error_df['cmdb_id'].isin(frontends)) &
    (error_df['dt'] >= start_dt) &
    (error_df['dt'] < end_dt)
].copy()

# Prepare message column as string (preserve full text)
error_front_filtered['message_str'] = error_front_filtered['message'].astype(str)

# 1) Distinct messages table: message, count, earliest_timestamp (top 20 by count desc)
msg_agg = (
    error_front_filtered
    .groupby('message_str', dropna=False)
    .agg(count=('message_str', 'size'), earliest_dt=('dt', 'min'))
    .reset_index()
    .sort_values(by=['count', 'earliest_dt'], ascending=[False, True])
)
msg_agg['earliest_timestamp'] = msg_agg['earliest_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
distinct_messages_top20 = msg_agg[['message_str', 'count', 'earliest_timestamp']].head(20).rename(columns={'message_str':'message'})

# 2) 'failed to retrieve' samples (case-insensitive), up to 20 earliest rows
mask_failed = error_front_filtered['message_str'].str.contains('failed to retrieve', case=False, na=False)
failed_samples = error_front_filtered[mask_failed].sort_values('dt', ascending=True).head(20).copy()
# compact columns: timestamp, cmdb_id, full message
failed_samples['timestamp'] = failed_samples['dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
failed_samples_compact = failed_samples[['timestamp', 'cmdb_id', 'message_str']].rename(columns={'message_str':'message'})

# Store results in kernel variables for later steps:
error_front_filtered, distinct_messages_top20, failed_samples_compact

```
Out[9]:
```
Summary — error_logs for frontends (compact)

- Total filtered error rows (frontends, 09:00–09:30): 558 occurrences.
- Distinct messages: effectively a single repeating message — "severity: warning, message: failed to retrieve..." — count = 558, earliest occurrence = 2022-03-20 09:00:01+08:00.
- Sample rows (earliest examples): messages appear from 09:00:01 onward across frontend-0 and frontend-2 (e.g., 09:00:01 frontend-0, 09:00:01 frontend-2), then repeat every few seconds/minutes and include frontend-1 starting at ~09:00:55; samples show consistent identical warning text.
- Interpretation: the error log is dominated by a single, repeated warning ("failed to retrieve") across all three frontends throughout the incident window — a persistent, system-wide symptom rather than many distinct error types.
- Recommended next steps: investigate what component or backend the "failed to retrieve" refers to (full message details), correlate these timestamps with trace/metric spikes (especially around 09:00–09:02) and backend/service targets to find the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(      timestamp     cmdb_id                                            message                        dt                                        message_str
0    1647738001  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 09:00:01+08:00  severity: warning, message: failed to retrieve...
1    1647738001  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 09:00:01+08:00  severity: warning, message: failed to retrieve...
2    1647738007  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 09:00:07+08:00  severity: warning, message: failed to retrieve...
3    1647738007  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 09:00:07+08:00  severity: warning, message: failed to retrieve...
4    1647738013  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 09:00:13+08:00  severity: warning, message: failed to retrieve...
..          ...         ...                                                ...                       ...                                                ...
553  1647739785  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 09:29:45+08:00  severity: warning, message: failed to retrieve...
554  1647739788  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 09:29:48+08:00  severity: warning, message: failed to retrieve...
555  1647739793  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 09:29:53+08:00  severity: warning, message: failed to retrieve...
556  1647739795  frontend-1  severity: warning, message: failed to retrieve... 2022-03-20 09:29:55+08:00  severity: warning, message: failed to retrieve...
557  1647739796  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 09:29:56+08:00  severity: warning, message: failed to retrieve...

[558 rows x 5 columns],                                              message  count        earliest_timestamp
0  severity: warning, message: failed to retrieve...    558  2022-03-20 09:00:01+0800,                    timestamp     cmdb_id                                            message
0   2022-03-20 09:00:01+0800  frontend-0  severity: warning, message: failed to retrieve...
1   2022-03-20 09:00:01+0800  frontend-2  severity: warning, message: failed to retrieve...
2   2022-03-20 09:00:07+0800  frontend-0  severity: warning, message: failed to retrieve...
3   2022-03-20 09:00:07+0800  frontend-2  severity: warning, message: failed to retrieve...
4   2022-03-20 09:00:13+0800  frontend-2  severity: warning, message: failed to retrieve...
5   2022-03-20 09:00:15+0800  frontend-0  severity: warning, message: failed to retrieve...
6   2022-03-20 09:00:17+0800  frontend-2  severity: warning, message: failed to retrieve...
7   2022-03-20 09:00:19+0800  frontend-0  severity: warning, message: failed to retrieve...
8   2022-03-20 09:00:24+0800  frontend-2  severity: warning, message: failed to retrieve...
9   2022-03-20 09:00:29+0800  frontend-2  severity: warning, message: failed to retrieve...
10  2022-03-20 09:00:33+0800  frontend-2  severity: warning, message: failed to retrieve...
11  2022-03-20 09:00:38+0800  frontend-0  severity: warning, message: failed to retrieve...
12  2022-03-20 09:00:39+0800  frontend-2  severity: warning, message: failed to retrieve...
13  2022-03-20 09:00:44+0800  frontend-2  severity: warning, message: failed to retrieve...
14  2022-03-20 09:00:52+0800  frontend-2  severity: warning, message: failed to retrieve...
15  2022-03-20 09:00:55+0800  frontend-1  severity: warning, message: failed to retrieve...
16  2022-03-20 09:00:58+0800  frontend-0  severity: warning, message: failed to retrieve...
17  2022-03-20 09:01:00+0800  frontend-2  severity: warning, message: failed to retrieve...
18  2022-03-20 09:01:07+0800  frontend-1  severity: warning, message: failed to retrieve...
19  2022-03-20 09:01:08+0800  frontend-0  severity: warning, message: failed to retrieve...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# Reuse existing trace_df and tz if available
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Target cmdb_ids and keyword filter
targets = [
    'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2',
    'adservice-0', 'adservice-1', 'adservice-2'
]
keywords = ['error_rate', 'duration_p95', 'duration_mean', 'row_count']
pattern = '|'.join(keywords)

# 1) Compute global P95 and total points per (cmdb_id, trace_name) using full trace_df (global thresholds)
grp_full = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_global_p95 = grp_full.quantile(0.95)
trace_total_points = grp_full.size()

df_trace_global_full = (
    pd.DataFrame({
        'global_p95': trace_global_p95,
        'total_points_in_full_series': trace_total_points
    })
    .reset_index()
)

# 2) Filter trace records to targets and incident subwindow 09:00-09:05 UTC+8
start_dt = tz.localize(pd.Timestamp('2022-03-20 09:00:00'))
end_dt = tz.localize(pd.Timestamp('2022-03-20 09:05:00'))  # exclusive

# ensure dt column exists and is timezone-aware
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

trace_incident = trace_df[
    (trace_df['cmdb_id'].isin(targets)) &
    (trace_df['dt'] >= start_dt) &
    (trace_df['dt'] < end_dt) &
    (trace_df['trace_name'].str.contains(pattern, regex=True))
].copy()

# 3) Prepare df_pairs (global thresholds) for the targeted (cmdb_id, trace_name) pairs (thresholds computed globally)
df_pairs = df_trace_global_full[
    (df_trace_global_full['cmdb_id'].isin(targets)) &
    (df_trace_global_full['trace_name'].str.contains(pattern, regex=True))
].copy()

# 4) Join global thresholds to incident rows (inner join ensures only pairs present in global thresholds are considered)
trace_incident_join = trace_incident.merge(df_pairs, on=['cmdb_id', 'trace_name'], how='inner')

# Mark above-P95 entries and record dt_above
trace_incident_join['above_p95'] = trace_incident_join['value'] > trace_incident_join['global_p95']
trace_incident_join['dt_above'] = trace_incident_join['dt'].where(trace_incident_join['above_p95'], pd.NaT)

# 5) Aggregate incident stats per (cmdb_id, trace_name)
trace_incident_agg = (
    trace_incident_join
    .groupby(['cmdb_id', 'trace_name'])
    .agg(
        incident_points_count_above_p95 = ('above_p95', 'sum'),
        earliest_timestamp_above = ('dt_above', 'min'),
        max_value_in_incident_window = ('value', 'max'),
        incident_points_total = ('value', 'count')
    )
    .reset_index()
)

# 6) Merge aggregation back onto df_pairs to include series with zero incident rows
trace_result = df_pairs.merge(trace_incident_agg, on=['cmdb_id', 'trace_name'], how='left')

# Fill missing numeric values with 0 and format timestamps
trace_result['incident_points_count_above_p95'] = trace_result['incident_points_count_above_p95'].fillna(0).astype(int)
trace_result['incident_points_total'] = trace_result['incident_points_total'].fillna(0).astype(int)

def fmt_ts(x):
    if pd.isna(x):
        return None
    return x.strftime('%Y-%m-%d %H:%M:%S%z')

trace_result['earliest_timestamp_above'] = trace_result['earliest_timestamp_above'].apply(fmt_ts)

# 7) Select required compact columns and sort
final_cols = [
    'cmdb_id',
    'trace_name',
    'global_p95',
    'total_points_in_full_series',
    'incident_points_count_above_p95',
    'earliest_timestamp_above',
    'max_value_in_incident_window',
    'incident_points_total'
]
trace_compact = trace_result[final_cols].copy()

# Sorting by incident_points_count_above_p95 desc, then earliest timestamp asc (None last)
trace_compact['__earliest_sort'] = pd.to_datetime(trace_compact['earliest_timestamp_above'], format='%Y-%m-%d %H:%M:%S%z', utc=True, errors='coerce')
trace_compact = trace_compact.sort_values(by=['incident_points_count_above_p95', '__earliest_sort'], ascending=[False, True], na_position='last').drop(columns='__earliest_sort')

# Limit to top 20 rows for compact output
trace_compact_top20 = trace_compact.head(20)

# Store intermediates for later steps
df_trace_global_full, df_pairs, trace_incident, trace_incident_join, trace_incident_agg, trace_compact_top20

# Return only the compact table
trace_compact_top20

```
Out[10]:
```
Summary — trace P95 check for selected services in 2022-03-20 09:00–09:05 (UTC+8)

What I did: computed global P95 per (cmdb_id, trace_name) using the full trace.csv, then checked traces for productcatalogservice-0/1/2 and adservice-0/1/2 in the 5‑minute subwindow (09:00–09:05). I restricted trace_names to error_rate, duration_p95, duration_mean, or row_count.

Key findings (compact):
- Only a few trace series exceeded their global P95 during the 09:00–09:05 window; most series had 0 or 1 exceedance in that subwindow.
- Top series with 2 exceedances (incident_points_count_above_p95 = 2):
  - productcatalogservice-0 — trace.from_frontend-2.duration_mean: global_p95=0.004124, 2 points above P95, earliest exceed 09:00:00, max=0.004205, incident_points_total=5.
  - adservice-2 — trace.from_frontend-0.duration_mean: global_p95=0.000021, 2 points above P95, earliest exceed 09:01:00, max=0.000034, incident_points_total=5.
  - adservice-2 — trace.from_frontend-2.duration_mean: global_p95=0.000021, 2 points above P95, earliest exceed 09:02:00, max=0.000024, incident_points_total=5.
  - adservice-2 — trace.from_frontend-2.duration_p95: global_p95=0.000034, 2 points above P95, earliest exceed 09:02:00, max=0.000042, incident_points_total=5.
  - productcatalogservice-0 — trace.to/from checkoutservice traces and some from_frontend series show 1 exceedance each (earliest at 09:00–09:01).
- Several row_count metrics for productcatalogservice and others show single-point increases (e.g., row_count = 75 vs global p95 74.55) — small absolute jumps but flagged as above P95.
- Most series had 5 incident points (the 5-minute window); some checkout-related traces had fewer total points (e.g., 2–4).

Interpretation & next steps:
- The exceedances are brief and sparse (mostly 1–2 points per series) and concentrated around 09:00–09:02.
- Notable candidates for investigation are productcatalogservice-0 and adservice-2 traces (their duration_mean/duration_p95 series show the highest incident exceedances).
- I recommend inspecting full traces and request logs for those services at ~09:00–09:02 to see whether these brief latency spikes correlate with increased errors or backend issues.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  trace_name  global_p95  total_points_in_full_series  incident_points_count_above_p95  earliest_timestamp_above  max_value_in_incident_window  incident_points_total
56   productcatalogservice-0         trace.from_frontend-2.duration_mean    0.004124                           30                                2  2022-03-20 09:00:00+0800                      0.004205                      5
24               adservice-2         trace.from_frontend-0.duration_mean    0.000021                           30                                2  2022-03-20 09:01:00+0800                      0.000034                      5
32               adservice-2         trace.from_frontend-2.duration_mean    0.000021                           30                                2  2022-03-20 09:02:00+0800                      0.000024                      5
33               adservice-2          trace.from_frontend-2.duration_p95    0.000034                           30                                2  2022-03-20 09:02:00+0800                      0.000042                      5
44   productcatalogservice-0  trace.from_checkoutservice-2.duration_mean    0.004815                           21                                1  2022-03-20 09:00:00+0800                      0.005159                      4
45   productcatalogservice-0   trace.from_checkoutservice-2.duration_p95    0.004815                           21                                1  2022-03-20 09:00:00+0800                      0.005159                      4
48   productcatalogservice-0         trace.from_frontend-0.duration_mean    0.004077                           30                                1  2022-03-20 09:00:00+0800                      0.004092                      5
59   productcatalogservice-0             trace.from_frontend-2.row_count   74.550000                           30                                1  2022-03-20 09:00:00+0800                     75.000000                      5
139  productcatalogservice-2             trace.from_frontend-2.row_count   74.550000                           30                                1  2022-03-20 09:00:00+0800                     75.000000                      5
4                adservice-0         trace.from_frontend-1.duration_mean    0.000016                           30                                1  2022-03-20 09:01:00+0800                      0.000016                      5
15               adservice-1             trace.from_frontend-0.row_count   10.100000                           30                                1  2022-03-20 09:01:00+0800                     11.000000                      5
16               adservice-1         trace.from_frontend-1.duration_mean    0.000016                           30                                1  2022-03-20 09:01:00+0800                      0.000016                      5
17               adservice-1          trace.from_frontend-1.duration_p95    0.000021                           30                                1  2022-03-20 09:01:00+0800                      0.000023                      5
25               adservice-2          trace.from_frontend-0.duration_p95    0.000029                           30                                1  2022-03-20 09:01:00+0800                      0.000131                      5
27               adservice-2             trace.from_frontend-0.row_count   10.100000                           30                                1  2022-03-20 09:01:00+0800                     11.000000                      5
36   productcatalogservice-0  trace.from_checkoutservice-0.duration_mean    0.005304                           19                                1  2022-03-20 09:01:00+0800                      0.005317                      2
84   productcatalogservice-1  trace.from_checkoutservice-2.duration_mean    0.005219                           18                                1  2022-03-20 09:01:00+0800                      0.006486                      3
85   productcatalogservice-1   trace.from_checkoutservice-2.duration_p95    0.005500                           18                                1  2022-03-20 09:01:00+0800                      0.006486                      3
88   productcatalogservice-1         trace.from_frontend-0.duration_mean    0.004040                           30                                1  2022-03-20 09:01:00+0800                      0.004172                      5
89   productcatalogservice-1          trace.from_frontend-0.duration_p95    0.005403                           30                                1  2022-03-20 09:01:00+0800                      0.005803                      5```
```